In [1]:
# Install PySpark
!pip install pyspark==3.5.1 findspark -q

# =========================================================
# Initialize Spark
# =========================================================
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = (SparkSession.builder \
    .master("local[*]") \
    .appName("Ecommerce Project") \
    .config("spark.sql.legacy.timeParserPolicy","LEGACY") \
    .getOrCreate())

print("Spark Version:", spark.version)

#  Upload Dataset
from google.colab import files
uploaded = files.upload()

# Get the name of the uploaded file
file_name = list(uploaded.keys())[0]

df = spark.read.csv(f"/content/{file_name}", # Corrected file path
                    header=True,
                    inferSchema=True,
                    encoding='ISO-8859-1')

df.show(5)
# 15. Basic Data Analysis

df.printSchema()
df.describe().show()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 12.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
Spark Version: 3.5.1


Saving sample (1).csv to sample (1).csv
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|   

**CONVERTING DATA COLUMNS**





In [2]:
from pyspark.sql import functions as F
df = df.withColumn(
    "Order Date",
    F.to_date(F.col("Order Date"), "MM/dd/yyyy")
)

df = df.withColumn(
    "Ship Date",
    F.to_date(F.col("Ship Date"), "MM/dd/yyyy")
)
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [3]:
from pyspark.sql.functions import col, year, month, dayofweek, date_format
df = df.withColumn("Order Year", year(col("Order Date"))) \
       .withColumn("Order Month", month(col("Order Date"))) \
       .withColumn("Day Of Week", date_format(col("Order Date"), "EEEE"))
df.show()
print(df.columns)

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+----------+-----------+-----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|Order Year|Order Month|Day Of Week|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+----------+-----------+-----------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|       Cla

**MONTHLY SALES ANALYSIS**

In [4]:
from pyspark.sql.functions import sum as _sum
import plotly.express as px
monthly_sales = df.groupBy("Order Month") \
                  .agg(_sum("Sales").alias("Total Sales")) \
                  .orderBy("Order Month")
monthly_sales.show()

# Convert to Pandas for Plotly
monthly_sales_pd = monthly_sales.toPandas()

# -----------------------------
# -----------------------------
fig = px.line(
    monthly_sales_pd,
    x="Order Month",
    y="Total Sales",
    title="Monthly Sales Trend",
    labels={"Order Month": "Month", "Total Sales": "Sales"}
)

fig.update_traces(mode="lines+markers")
fig.update_layout(template="plotly_white", xaxis_tickangle=45)

fig.show()

+-----------+------------------+
|Order Month|       Total Sales|
+-----------+------------------+
|          1| 94539.34159999999|
|          2| 59012.82540000001|
|          3|203719.26379999987|
|          4|135387.35759999996|
|          5| 153513.3096999999|
|          6|151039.43330000006|
|          7| 145623.8500000002|
|          8|157642.25099999984|
|          9| 303536.6657000002|
|         10| 198440.0027000002|
|         11| 348834.5570000001|
|         12| 321160.9985000003|
+-----------+------------------+



**SALES BY CATEGORY**

In [5]:

# -------------------------
#  Category Analysis
# -------------------------
Sale_By_Category = df.groupBy("Category") \
                     .agg(_sum("Sales").alias("Total_Sales"))
Sale_By_Category.show()

category_df = Sale_By_Category.toPandas()

fig = px.pie(
    category_df,
    values="Total_Sales",
    names="Category",
    title="Sales Distribution by Category"
)
fig.show()



+---------------+-----------------+
|       Category|      Total_Sales|
+---------------+-----------------+
|Office Supplies|703502.9280000031|
|      Furniture|733046.8612999996|
|     Technology|835900.0669999964|
+---------------+-----------------+



**SALES BY SUB-CATEGORY**

In [6]:
Sale_By_SubCategory = df.groupBy("Sub-Category") \
                  .agg(_sum("Sales").alias("Total_Sales")) \
                  .orderBy("Sub-Category")
Sale_By_SubCategory.show()

#import plotly.express as px

# Convert Spark DataFrame to Pandas
sub_category_pandas_df = Sale_By_SubCategory.toPandas()

# Create interactive bar chart
fig = px.bar(
    sub_category_pandas_df,
    x="Sub-Category",
    y="Total_Sales",
    title="Sales by Sub-Category",
    labels={
        "Sub-Category": "Sub-Category",
        "Total_Sales": "Total Sales"
    },
        # similar to steelblue look
)
fig.update_layout(title_font_size=30,font=dict(size=20),template="plotly_white")
fig.show()

+------------+------------------+
|Sub-Category|       Total_Sales|
+------------+------------------+
| Accessories| 167380.3180000001|
|  Appliances|        107532.161|
|         Art|27118.791999999954|
|     Binders|199905.71700000006|
|   Bookcases|114879.99629999997|
|      Chairs|328449.10300000076|
|     Copiers|149528.02999999994|
|   Envelopes|15339.489999999993|
|   Fasteners|3008.6559999999995|
| Furnishings| 82752.23000000004|
|      Labels|         12486.312|
|    Machines|189238.63099999996|
|       Paper| 75356.11799999999|
|      Phones| 329753.0880000001|
|     Storage|216803.21200000012|
|    Supplies| 45952.47000000001|
|      Tables| 206965.5320000001|
+------------+------------------+



**MONTHLY PROFIT ANALYSIS**

In [7]:
#from pyspark.sql.functions import sum as _sum
import plotly.express as px
# Group by Order Month and calculate total profit
profit_by_month = df.groupBy("Order Month") \
                    .agg(_sum("Profit").alias("Total_Profit")) \
                    .orderBy("Order Month")

# Show Spark result
profit_by_month.show()
pandas_profit = profit_by_month.toPandas()

# #import plotly.express as px
fig = px.line(
    pandas_profit,
    x="Order Month",
    y="Total_Profit",
    markers=True,
    title="Monthly Profit Trend"
)

fig.update_traces(
    line=dict(width=3),
    marker=dict(
        size=8,
        color=[
            "green" if val >= 0 else "red"
            for val in pandas_profit["Total_Profit"]
        ]
    )
)

fig.update_layout(xaxis_tickangle=-45, template="plotly_white")

fig.show()

+-----------+------------------+
|Order Month|      Total_Profit|
+-----------+------------------+
|          1| 9135.780299999999|
|          2|10196.932399999994|
|          3| 28359.30890000002|
|          4|11849.297500000002|
|          5|          22376.71|
|          6|21327.156000000025|
|          7| 13929.99099999998|
|          8|21745.858500000006|
|          9|36270.184900000044|
|         10|31703.218000000037|
|         11| 35412.80679999993|
|         12| 43400.35790000003|
+-----------+------------------+



**PROFIT BY CATEGORY**

In [8]:
Profit_By_Category = df.groupBy("Category") \
                  .agg(_sum("Profit").alias("Profit_Category")) \
                  .orderBy("Category")
Profit_By_Category.show()
#Convert to Pandas (after aggregation)
pandas_profit_cat = Profit_By_Category.toPandas()
#Create Interactive Pie Chart
#import plotly.express as px

fig = px.pie(
    pandas_profit_cat,
    values="Profit_Category",
    names="Category",
    title="Profit Distribution by Category",
    hole=0.5,
    color_discrete_sequence=["#1f77b4", "#ff7f0e", "#2ca02c"]  # custom colors
)

fig.update_traces(
    textposition="inside",
    textinfo="percent+label"
)

fig.update_layout(title_font_size=30,font=dict(size=20),template="plotly_white")

fig.show()

+---------------+------------------+
|       Category|   Profit_Category|
+---------------+------------------+
|      Furniture| 19686.42720000003|
|Office Supplies|120632.87839999991|
|     Technology|145388.29659999989|
+---------------+------------------+



**PROFIT BY SUB-CATEGORY**

In [9]:
Profit_By_SubCategory = df.groupBy("Sub-Category") \
                          .agg(_sum("Profit").alias("Profit_Sub-Category")) \
                          .orderBy("Sub-Category")

Profit_By_SubCategory.show()
pandas_profit_subcat = Profit_By_SubCategory.toPandas()
#import plotly.express as px

fig = px.bar(
    pandas_profit_subcat,
    x="Sub-Category",
    y="Profit_Sub-Category",
    title="Profit by Sub-Category",
    color="Profit_Sub-Category",                    # Color intensity based on profit
    color_continuous_scale="RdYlGn",              # Red = low profit, Green = high profit
    labels={"Profit_Sub-Category": "Profit"}
)

fig.update_layout(
    xaxis_tickangle=-45,
    template="plotly_white"
)

fig.show()

+------------+-------------------+
|Sub-Category|Profit_Sub-Category|
+------------+-------------------+
| Accessories|  41936.63569999993|
|  Appliances| 18138.005399999995|
|         Art|  6527.786999999998|
|     Binders| 30038.821299999996|
|   Bookcases|-3472.5559999999978|
|      Chairs| 26590.166300000026|
|     Copiers|  55617.82490000001|
|   Envelopes|  6461.269100000003|
|   Fasteners|  942.6377999999997|
| Furnishings| 14294.297999999995|
|      Labels|  5546.253999999998|
|    Machines|          3384.7569|
|       Paper|  32795.56099999999|
|      Phones|         44449.0791|
|     Storage|         21529.9083|
|    Supplies|-1347.3654999999983|
|      Tables|-17725.481100000008|
+------------+-------------------+



**SALES VS PROFIT BY COUNTRY,REGION AND CITY**

In [10]:
from pyspark.sql.functions import sum as _sum

# By Country
Sales_Profit_Country_df = df.groupBy("Country") \
    .agg(
        _sum("Sales").alias("Total_Sales"),
        _sum("Profit").alias("Total_Profit")
    ) \
    .orderBy("Country")

Sales_Profit_Country_df.show()

# By Region
Sales_Profit_Region_df = df.groupBy("Region") \
    .agg(
        _sum("Sales").alias("Total_Sales"),
        _sum("Profit").alias("Total_Profit")
    ) \
    .orderBy("Region") # Changed to orderBy("Region")

Sales_Profit_Region_df.show()

# By City
Sales_Profit_City_df = df.groupBy("City") \
    .agg(
        _sum("Sales").alias("Total_Sales"),
        _sum("Profit").alias("Total_Profit")
    ) \
    .orderBy("City") # Changed to orderBy("City")

Sales_Profit_City_df.show()

# Convert to Pandas DataFrames
pandas_country = Sales_Profit_Country_df.toPandas()
pandas_region = Sales_Profit_Region_df.toPandas()
pandas_city = Sales_Profit_City_df.toPandas()

import plotly.graph_objects as go

# Plot for Country
fig_country = go.Figure()

fig_country.add_trace(go.Bar(
    x=pandas_country["Country"],
    y=pandas_country["Total_Sales"],
    name="Total Sales",
    marker_color="steelblue"
))

fig_country.add_trace(go.Bar(
    x=pandas_country["Country"],
    y=pandas_country["Total_Profit"],
    name="Total Profit",
    marker_color="green"
))

fig_country.update_layout(
    title="Sales vs Profit by Country",
    xaxis_title="Country",
    yaxis_title="Amount ($)",
    barmode="group",
    template="plotly_white"
)

fig_country.show()

# Plot for Region
fig_region = go.Figure()

fig_region.add_trace(go.Bar(
    x=pandas_region["Region"],
    y=pandas_region["Total_Sales"],
    name="Total Sales",
    marker_color="steelblue"
))

fig_region.add_trace(go.Bar(
    x=pandas_region["Region"],
    y=pandas_region["Total_Profit"],
    name="Total Profit",
    marker_color="green"
))

fig_region.update_layout(
    title="Sales vs Profit by Region", # Corrected title
    xaxis_title="Region",
    yaxis_title="Amount ($)",
    barmode="group",
    template="plotly_white"
)

fig_region.show()

# Plot for City
fig_city = go.Figure()

fig_city.add_trace(go.Bar(
    x=pandas_city["City"],
    y=pandas_city["Total_Sales"],
    name="Total Sales",
    marker_color="steelblue"
))

fig_city.add_trace(go.Bar(
    x=pandas_city["City"],
    y=pandas_city["Total_Profit"],
    name="Total Profit",
    marker_color="green"
))

fig_city.update_layout(
    title="Sales vs Profit by City", # Corrected title
    xaxis_title="City",
    yaxis_title="Amount ($)",
    barmode="group",
    template="plotly_white"
)

fig_city.show()

+-------------+------------------+------------------+
|      Country|       Total_Sales|      Total_Profit|
+-------------+------------------+------------------+
|United States|2272449.8562999545|285707.60220000165|
+-------------+------------------+------------------+

+-------+------------------+------------------+
| Region|       Total_Sales|      Total_Profit|
+-------+------------------+------------------+
|Central| 497800.8728000007| 40150.50299999999|
|   East| 672194.0539999981| 91603.05670000015|
|  South|388983.58500000037|46650.341000000044|
|   West| 713471.3445000004|107303.70150000004|
+-------+------------------+------------------+

+-----------------+------------------+------------------+
|             City|       Total_Sales|      Total_Profit|
+-----------------+------------------+------------------+
|         Aberdeen|              25.5|              6.63|
|          Abilene|             1.392|           -3.7584|
|            Akron|2729.9860000000003|         -186.63

**CUSTOMER SEGMENT DISTRIBUTION**

In [11]:
# =====================================================
# Imports
# =====================================================

from pyspark.sql.functions import count
import plotly.express as px

# =====================================================
# Count Customers by Segment
# =====================================================

segment_count_df = df.groupBy("Segment").agg(
    count("Customer ID").alias("Customer_Count")
)

# Convert Spark → Pandas
segment_count_pd = segment_count_df.toPandas()

# =====================================================
# Bar Chart
# =====================================================

fig = px.bar(
    segment_count_pd,
    x="Segment",
    y="Customer_Count",
    color="Segment",
    text="Customer_Count",
    title="Customer Distribution by Segment"
)

fig.update_layout(
    template="plotly_white",
    height=500,
    width=800
)

fig.show()

**CUSTOMER SEGMENT (SALES VS PROFIT)**

In [12]:
# =====================================================
# Imports
# =====================================================

from pyspark.sql.functions import sum as _sum
import plotly.graph_objects as go

# =====================================================
# Aggregate Sales & Profit by Segment
# =====================================================

segment_df = df.groupBy("Segment").agg(
    _sum("Sales").alias("Total_Sales"),
    _sum("Profit").alias("Total_Profit")
)

# Convert Spark → Pandas
segment_pd = segment_df.toPandas()

# =====================================================
# Profit Margin %
# =====================================================

segment_pd["Profit_Margin_%"] = (
    segment_pd["Total_Profit"] / segment_pd["Total_Sales"]
) * 100

# =====================================================
# All-in-One Chart
# =====================================================

fig = go.Figure()

# Sales Bars
fig.add_trace(
    go.Bar(
        x=segment_pd["Segment"],
        y=segment_pd["Total_Sales"],
        name="Sales",
        marker_color="steelblue"
    )
)

# Profit Bars
fig.add_trace(
    go.Bar(
        x=segment_pd["Segment"],
        y=segment_pd["Total_Profit"],
        name="Profit",
        marker_color="green"
    )
)

# Profit Margin Line
fig.add_trace(
    go.Scatter(
        x=segment_pd["Segment"],
        y=segment_pd["Profit_Margin_%"],
        mode="lines+markers",
        name="Profit Margin %",
        yaxis="y2",
        line=dict(color="red", width=3)
    )
)

# =====================================================
# Layout
# =====================================================

fig.update_layout(

title="Customer Segment Performance (Sales, Profit & Profit Margin)",

xaxis=dict(title="Customer Segment"),

yaxis=dict(
    title="Sales / Profit"
),

yaxis2=dict(
    title="Profit Margin %",
    overlaying="y",
    side="right"
),

barmode="group",

template="plotly_white",
height=500,
width=900
)

fig.show()

# **DASHBOARD**

In [13]:
# =====================================================
# Remove Warnings
# =====================================================
import warnings
warnings.filterwarnings("ignore")

# =====================================================
# Imports
# =====================================================
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyspark.sql.functions import sum as _sum

# =====================================================
# Convert Spark DataFrames to Pandas
# =====================================================
sales_cat = Sale_By_Category.toPandas()
sales_subcat = Sale_By_SubCategory.toPandas()
monthly_sales_pd = monthly_sales.toPandas()

country_pd = Sales_Profit_Country_df.toPandas()
region_pd = Sales_Profit_Region_df.toPandas()
city_pd = Sales_Profit_City_df.toPandas()

profit_cat_pd = pandas_profit_cat.copy()
profit_subcat_pd = pandas_profit_subcat.copy()

# =====================================================
# Monthly Profit Calculation (FIXED)
# =====================================================
profit_month_pd = df.groupBy("Order Month").agg(
    _sum("Profit").alias("Total_Profit")
).toPandas()

# Convert month number to month name
profit_month_pd["Order Month Name"] = pd.to_datetime(
    profit_month_pd["Order Month"],
    format="%m",
    errors="coerce"
).dt.strftime("%b")

monthly_sales_pd["Order Month Name"] = pd.to_datetime(
    monthly_sales_pd["Order Month"],
    format="%m",
    errors="coerce"
).dt.strftime("%b")

# =====================================================
# Aggregate Monthly Sales & Profit
# =====================================================
monthly_sales_pd = monthly_sales_pd.groupby(
    "Order Month Name", as_index=False
)["Total Sales"].sum()

profit_month_pd = profit_month_pd.groupby(
    "Order Month Name", as_index=False
)["Total_Profit"].sum()

# =====================================================
# Month Order
# =====================================================
month_order = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]

monthly_sales_pd["Order Month Name"] = pd.Categorical(
    monthly_sales_pd["Order Month Name"],
    categories=month_order,
    ordered=True
)

profit_month_pd["Order Month Name"] = pd.Categorical(
    profit_month_pd["Order Month Name"],
    categories=month_order,
    ordered=True
)

monthly_sales_pd = monthly_sales_pd.sort_values("Order Month Name")
profit_month_pd = profit_month_pd.sort_values("Order Month Name")

# =====================================================
# Customer Segment Aggregation
# =====================================================
segment_pd = df.groupBy("Segment").agg(
    _sum("Sales").alias("Total_Sales_Segment"),
    _sum("Profit").alias("Total_Profit_Segment")
).toPandas()

# =====================================================
# Top 20 Cities
# =====================================================
city_pd = city_pd.sort_values("Total_Sales",ascending=False).head(20)

# =====================================================
# KPI Calculations
# =====================================================
total_sales = monthly_sales_pd["Total Sales"].sum()

# Correct total profit calculation
total_profit = df.select(_sum("Profit")).toPandas().iloc[0,0]

total_categories = sales_cat["Category"].nunique()

# =====================================================
# Dashboard Layout
# =====================================================
fig = make_subplots(

rows=5,
cols=3,

subplot_titles=(

"", "", "",

"Monthly Sales Trend",
"Sales by Category",
"Sales by SubCategory",

"Monthly Profit Trend",
"Profit by Category",
"Profit by SubCategory",

"Sales vs Profit by Country",
"Sales vs Profit by Region",
"Top 20 Cities Sales vs Profit",

"Customer Segment Sales vs Profit",
"",
""

),

specs=[

[{"type":"indicator"},{"type":"indicator"},{"type":"indicator"}],

[{"type":"xy"},{"type":"domain"},{"type":"xy"}],

[{"type":"xy"},{"type":"domain"},{"type":"xy"}],

[{"type":"xy"},{"type":"xy"},{"type":"xy"}],

[{"type":"xy"},None,None]

]

)

# =====================================================
# KPI CARDS
# =====================================================
fig.add_trace(go.Indicator(
mode="number",
value=total_sales,
number={'prefix':"$",'valueformat':',.0f'},
title={"text":"Total Sales"}
),row=1,col=1)

fig.add_trace(go.Indicator(
mode="number",
value=total_profit,
number={'prefix':"$",'valueformat':',.0f'},
title={"text":"Total Profit"}
),row=1,col=2)

fig.add_trace(go.Indicator(
mode="number",
value=total_categories,
title={"text":"Total Categories"}
),row=1,col=3)

# =====================================================
# SALES ANALYSIS
# =====================================================
fig.add_trace(go.Scatter(
x=monthly_sales_pd["Order Month Name"],
y=monthly_sales_pd["Total Sales"],
mode="lines+markers",
line=dict(color="#1f77b4",width=3),
showlegend=False
),row=2,col=1)

fig.add_trace(go.Pie(
labels=sales_cat["Category"],
values=sales_cat["Total_Sales"],
hole=0.4,
showlegend=False
),row=2,col=2)

fig.add_trace(go.Bar(
x=sales_subcat["Sub-Category"],
y=sales_subcat["Total_Sales"],
marker_color="#1f77b4",
showlegend=False
),row=2,col=3)

# =====================================================
# PROFIT ANALYSIS
# =====================================================
fig.add_trace(go.Scatter(
x=profit_month_pd["Order Month Name"],
y=profit_month_pd["Total_Profit"],
mode="lines+markers",
line=dict(color="#2ca02c",width=4),
marker=dict(size=8),
showlegend=False
),row=3,col=1)

fig.add_trace(go.Pie(
labels=profit_cat_pd["Category"],
values=profit_cat_pd["Profit_Category"],
hole=0.4,
showlegend=False
),row=3,col=2)

fig.add_trace(go.Bar(
x=profit_subcat_pd["Sub-Category"],
y=profit_subcat_pd["Profit_Sub-Category"],
marker_color="#2ca02c",
showlegend=False
),row=3,col=3)

# =====================================================
# GEOGRAPHY ANALYSIS
# =====================================================
fig.add_trace(go.Bar(
x=country_pd["Country"],
y=country_pd["Total_Sales"],
name="Sales",
marker_color="#1f77b4"
),row=4,col=1)

fig.add_trace(go.Bar(
x=country_pd["Country"],
y=country_pd["Total_Profit"],
name="Profit",
marker_color="#2ca02c"
),row=4,col=1)

fig.add_trace(go.Bar(
x=region_pd["Region"],
y=region_pd["Total_Sales"],
marker_color="#1f77b4",
showlegend=False
),row=4,col=2)

fig.add_trace(go.Bar(
x=region_pd["Region"],
y=region_pd["Total_Profit"],
marker_color="#2ca02c",
showlegend=False
),row=4,col=2)

fig.add_trace(go.Bar(
x=city_pd["City"],
y=city_pd["Total_Sales"],
marker_color="#1f77b4",
showlegend=False
),row=4,col=3)

fig.add_trace(go.Bar(
x=city_pd["City"],
y=city_pd["Total_Profit"],
marker_color="#2ca02c",
showlegend=False
),row=4,col=3)

# =====================================================
# CUSTOMER SEGMENT
# =====================================================
fig.add_trace(go.Bar(
x=segment_pd["Segment"],
y=segment_pd["Total_Sales_Segment"],
marker_color="steelblue",
showlegend=False
),row=5,col=1)

fig.add_trace(go.Bar(
x=segment_pd["Segment"],
y=segment_pd["Total_Profit_Segment"],
marker_color="#2ca02c",
showlegend=False
),row=5,col=1)

# =====================================================
# Layout
# =====================================================
fig.update_layout(

title={
"text":"<b>Ecommerce Sales & Profit Analytics Dashboard</b>",
"x":0.5,
"xanchor":"center",
"font":dict(size=32)
},

height=1500,
width=1700,

barmode="group",
template="plotly_white"

)

fig.update_xaxes(tickangle=45,row=4,col=3)

fig.show()

**SALES TO PROFIT RATIO - CUSTOMER SEGMENT**

In [14]:
from pyspark.sql.functions import sum as _sum, round as _round, col
import plotly.graph_objects as go

# Aggregate data
Sales_Profit_Customer_Segment = df.groupBy("Segment").agg(
    _sum("Sales").alias("Total_Sales"),
    _sum("Profit").alias("Total_Profit")
)

# Add ratio column
Sales_Profit_Customer_Segment = Sales_Profit_Customer_Segment.withColumn(
    "Sales_Profit_Ratio",
    _round(col("Total_Sales") / col("Total_Profit"), 2)
)

# Convert to Pandas
ratio_pd = Sales_Profit_Customer_Segment.toPandas()

# Sort values
ratio_pd = ratio_pd.sort_values(by="Sales_Profit_Ratio", ascending=False)

# Create figure
fig = go.Figure()

# Sales bar
fig.add_trace(
    go.Bar(
        x=ratio_pd["Segment"],
        y=ratio_pd["Total_Sales"],
        name="Sales",
        marker_color="blue"
    )
)

# Profit bar
fig.add_trace(
    go.Bar(
        x=ratio_pd["Segment"],
        y=ratio_pd["Total_Profit"],
        name="Profit",
        marker_color="green"
    )
)



# Layout with secondary y-axis
fig.update_layout(
    title="Sales vs Profit by Segment with Ratio",
    xaxis=dict(title="Segment"),
    yaxis=dict(title="Sales / Profit"),
    yaxis2=dict(
        title="Sales / Profit Ratio",
        overlaying="y",
        side="right"
    ),
    barmode="group"
)

fig.show()

**RFM (Recency, Frequency, Monetary)**

In [15]:
from pyspark.sql.functions import max as _max, datediff, col, lit, count, sum as _sum
import plotly.express as px

max_date = df.select(_max("Order Date")).collect()[0][0]

RFM_df = df.groupBy("Customer ID").agg(
    _max("Order Date").alias("Last_Purchase_Date"),
    count("Order ID").alias("Frequency"),
    _sum("Sales").alias("Monetary")
)

RFM_df = RFM_df.withColumn(
    "Recency",
    datediff(lit(max_date), col("Last_Purchase_Date"))
)
RFM_df.show()
# Convert to Pandas for Plotly
rfm_pdf = RFM_df.toPandas()

fig = px.scatter(
    rfm_pdf,
    x="Frequency",
    y="Monetary",
    color="Recency",
    size="Monetary",
    hover_data=["Customer ID"],
    title="Customer Segmentation (RFM)"
)

fig.show()

+-----------+------------------+---------+------------------+-------+
|Customer ID|Last_Purchase_Date|Frequency|          Monetary|Recency|
+-----------+------------------+---------+------------------+-------+
|   VW-21775|        2017-12-02|       18|          6112.702|     28|
|   PB-19210|        2016-02-11|        2|           132.738|    688|
|   RR-19315|        2017-12-11|        4|           615.932|     19|
|   EM-13960|        2017-08-01|        6|           933.704|    151|
|   MY-17380|        2017-12-01|       13|2242.6119999999996|     29|
|   MS-17530|        2016-10-22|        7|475.65599999999995|    434|
|   KH-16630|        2017-11-02|       17|          3918.966|     58|
|   BD-11500|        2017-11-25|       10|4408.2970000000005|     35|
|   SW-20275|        2017-11-30|        7|           1966.65|     30|
|   AH-10690|        2016-11-15|       23| 7888.294000000001|    410|
|   PH-18790|        2017-10-21|        2|           729.648|     70|
|   JF-15490|       

**CUSTOMER CHURN PROBABILITY**

In [ ]:
# =====================================================
# 1️⃣ Imports
# =====================================================

from pyspark.sql.functions import col, max as _max, count, sum as _sum, datediff, lit, when, to_date
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array
import plotly.express as px
import pandas as pd

# =====================================================
# 2️⃣ Convert Order Date
# =====================================================

df = df.withColumn("Order Date", to_date(col("Order Date"), "MM/dd/yyyy"))

# =====================================================
# 3️⃣ RFM Feature Engineering
# =====================================================

max_order_date = df.agg(_max("Order Date")).collect()[0][0]

recency_df = df.groupBy("Customer ID").agg(
    _max("Order Date").alias("Last_Purchase_Date")
)

recency_df = recency_df.withColumn(
    "Recency",
    datediff(lit(max_order_date), col("Last_Purchase_Date"))
)

frequency_df = df.groupBy("Customer ID").agg(
    count("Order ID").alias("Frequency")
)

monetary_df = df.groupBy("Customer ID").agg(
    _sum("Sales").alias("Monetary")
)

rfm_df = recency_df.join(frequency_df,"Customer ID")\
                   .join(monetary_df,"Customer ID")

# =====================================================
# 4️⃣ Create Churn Label
# =====================================================

rfm_df = rfm_df.withColumn(
    "churn",
    when(col("Recency") > 250, 1).otherwise(0)
)

rfm_df.groupBy("churn").count().show()

# =====================================================
# 5️⃣ Feature Vector
# =====================================================

assembler = VectorAssembler(
    inputCols=["Recency","Frequency","Monetary"],
    outputCol="features"
)

data = assembler.transform(rfm_df)

# =====================================================
# 6️⃣ Scale Features
# =====================================================

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features"
)

scaler_model = scaler.fit(data)
scaled_data = scaler_model.transform(data)

# =====================================================
# 7️⃣ Train Test Split
# =====================================================

train,test = scaled_data.randomSplit([0.7,0.3],seed=42)

# =====================================================
# 8️⃣ Train Logistic Regression Model
# =====================================================

lr = LogisticRegression(
    featuresCol="scaled_features",
    labelCol="churn"
)

model = lr.fit(train)

# =====================================================
# 9️⃣ Predict Churn Probability
# =====================================================

predictions = model.transform(test)

predictions = predictions.withColumn(
    "Churn_Prob",
    vector_to_array(col("probability"))[1]
)

pred_pdf = predictions.select(
    "Customer ID",
    "Churn_Prob"
).toPandas()

# =====================================================
# 🔟 Prepare Curve Data
# =====================================================

pred_pdf = pred_pdf.sort_values("Churn_Prob").reset_index(drop=True)
pred_pdf["Customer_Rank"] = range(1,len(pred_pdf)+1)

# =====================================================
# 1️⃣1️⃣ Churn Probability Curve
# =====================================================

avg_churn = pred_pdf["Churn_Prob"].mean()

fig = px.line(
    pred_pdf,
    x="Customer_Rank",
    y="Churn_Prob",
    title="Customer Churn Probability Curve",
    labels={
        "Customer_Rank":"Customer Rank",
        "Churn_Prob":"Churn Probability"
    }
)

fig.update_traces(line=dict(width=3))

fig.add_hline(
    y=avg_churn,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Average Churn: {avg_churn:.2f}"
)

fig.update_layout(template="plotly_white")

fig.show()

# =====================================================
# 1️⃣2️⃣ Churn Risk Segmentation
# =====================================================

pred_pdf["Risk_Level"] = pred_pdf["Churn_Prob"].apply(
    lambda x: "Low Risk" if x < 0.3 else
              "Medium Risk" if x < 0.6 else
              "High Risk"
)

risk_summary = pred_pdf["Risk_Level"].value_counts().reset_index()
risk_summary.columns=["Risk_Level","Customers"]

fig2 = px.bar(
    risk_summary,
    x="Risk_Level",
    y="Customers",
    color="Risk_Level",
    title="Customer Churn Risk Segmentation",
    color_discrete_map={
        "Low Risk":"green",
        "Medium Risk":"orange",
        "High Risk":"red"
    }
)

fig2.update_layout(template="plotly_white")

fig2.show()

+-----+-----+
|churn|count|
+-----+-----+
|    1|  143|
|    0|  650|
+-----+-----+



**COMBINING RFM WITH CHURN PROBABILITY**

In [ ]:
from pyspark.sql.functions import when, col, max as _max, datediff, lit, count, sum as _sum
import plotly.express as px
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array
import pandas as pd

# Ensure SparkSession and df are available (assuming they are from previous cells)
# If not, add initialization here:
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("FixCell").getOrCreate()
# Assuming 'df' (Spark DataFrame) is available globally.

# ----------------------------- CUSTOMER FEATURES (RFM) -----------------------------
# Re-create RFM_df if it's not guaranteed to be in scope from previous cells
max_date = df.select(_max("Order Date")).collect()[0][0]

recency_df = df.groupBy("Customer ID").agg(
    _max("Order Date").alias("Last_Purchase_Date")
)
recency_df = recency_df.withColumn(
    "Recency",
    datediff(lit(max_date), col("Last_Purchase_Date"))
)

frequency_df = df.groupBy("Customer ID").agg(
    count("Order ID").alias("Frequency")
)

monetary_df = df.groupBy("Customer ID").agg(
    _sum("Sales").alias("Monetary")
)

# Combine RFM features into a single Spark DataFrame
customer_features = recency_df.join(frequency_df, "Customer ID") \
                               .join(monetary_df, "Customer ID")

# ----------------------------- CHURN PREDICTION (from cell 37woXShmduMq) -----------------------------
# Create churn label based on Recency
rfm_for_churn = customer_features.withColumn(
    "churn",
    when(col("Recency") > 250, 1).otherwise(0) # Using 250 days as churn threshold
)

# Feature Vector
assembler = VectorAssembler(
    inputCols=["Recency", "Frequency", "Monetary"],
    outputCol="features"
)
data_for_churn = assembler.transform(rfm_for_churn)

# Scale Features
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features"
)
scaler_model = scaler.fit(data_for_churn)
scaled_data_for_churn = scaler_model.transform(data_for_churn)

# Train Logistic Regression Model (on all data for probabilities)
lr = LogisticRegression(
    featuresCol="scaled_features",
    labelCol="churn"
)
model = lr.fit(scaled_data_for_churn) # Fit on all data to get probabilities for all customers

# Predict Churn Probability
predictions = model.transform(scaled_data_for_churn)
predictions_with_prob = predictions.withColumn(
    "Churn_Prob",
    vector_to_array(col("probability"))[1]
)

# ----------------------------- COMBINING RFM WITH CHURN PROBABILITY -----------------------------

# Convert necessary Spark DataFrames to Pandas
customer_features_pd = customer_features.toPandas()
predictions_pd = predictions_with_prob.select("Customer ID", "Churn_Prob").toPandas()

# Merge Churn_Prob with customer_features_pd
customer_features_pd = customer_features_pd.merge(
    predictions_pd[["Customer ID", "Churn_Prob"]],
    on="Customer ID",
    how="left"
)

# Define High/Low Value based on Monetary median
monetary_median = customer_features_pd["Monetary"].median() # Calculate median from pandas df
customer_features_pd["Value"] = customer_features_pd["Monetary"].apply(
    lambda x: "High-Value" if x >= monetary_median else "Low-Value"
)

# Define High/Low Risk based on Churn Probability
customer_features_pd["Risk"] = customer_features_pd["Churn_Prob"].apply(
    lambda x: "High-Risk" if x > 0.5 else "Low-Risk"
)

# Define Segment
customer_features_pd["Segment"] = customer_features_pd["Value"] + ", " + customer_features_pd["Risk"]

# Optional: preview segments
# print(customer_features_pd["Segment"].value_counts())

# ----------------------------- Plot Scatter: Monetary vs Frequency -----------------------------

color_map = {
    "High-Value, Low-Risk": "yellow",   # highlighted segment
    "High-Value, High-Risk": "red",
    "Low-Value, Low-Risk": "green",
    "Low-Value, High-Risk": "blue"
}

fig = px.scatter(
    customer_features_pd,
    x="Frequency",           # Number of orders
    y="Monetary",            # Total sales
    color="Segment",
    color_discrete_map=color_map,
    hover_data=["Customer ID", "Recency", "Churn_Prob"],
    title="Customer Segmentation: RFM + Churn Probability"
)

fig.update_traces(marker=dict(size=10, line=dict(width=1, color='black')))
fig.update_layout(template="plotly_white")

fig.show()


**DASHBOARD**

In [ ]:
# =====================================================
# Imports & Warnings
# =====================================================
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyspark.sql.functions import sum as _sum, count

# =====================================================
# Colors
# =====================================================
SALES_COLOR = "blue"
PROFIT_COLOR = "green"
MARGIN_COLOR = "red"
SEGMENT_COLORS = {"Consumer":"purple", "Corporate":"orange", "Home Office":"cyan"}
OTHER_PIE_COLORS = ["#FFD700","#FF6347","#20B2AA","#9370DB","#FF69B4","#8FBC8F"]

# RFM + Segment Colors
RFM_SEGMENT_COLORS = {
    "High-Value, Low-Risk": "yellow",
    "High-Value, High-Risk": "red",
    "Low-Value, Low-Risk": "green",
    "Low-Value, High-Risk": "blue"
}

# =====================================================
# Standardize columns & handle NA
# =====================================================
def standardize_columns(df, mapping):
    for col_name, new_name in mapping.items():
        if col_name in df.columns:
            df = df.rename(columns={col_name: new_name})
        else:
            df[new_name] = pd.NA
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].fillna(0)
        else:
            df[c] = df[c].fillna("")
    return df

# =====================================================
# Convert / Standardize Pandas DataFrames
# =====================================================
monthly_sales_pd = standardize_columns(monthly_sales.toPandas(), {"Order Month":"Order_Month", "Total Sales":"Total_Sales"})
profit_month_pd = standardize_columns(profit_by_month.toPandas(), {"Order Month":"Order_Month", "Total_Profit":"Total_Profit"})
sales_cat = standardize_columns(Sale_By_Category.toPandas(), {"Category":"Category", "Total_Sales":"Total_Sales"})
sales_subcat = standardize_columns(Sale_By_SubCategory.toPandas(), {"Sub-Category":"SubCategory", "Total_Sales":"Total_Sales"})
profit_cat_pd = standardize_columns(pandas_profit_cat, {"Category":"Category", "Profit_Category":"Total_Profit"})
profit_subcat_pd = standardize_columns(pandas_profit_subcat, {"Sub-Category":"SubCategory", "Profit_Sub-Category":"Total_Profit"})
country_pd = standardize_columns(Sales_Profit_Country_df.toPandas(), {"Country":"Country","Total_Sales":"Total_Sales", "Total_Profit":"Total_Profit"})
region_pd = standardize_columns(Sales_Profit_Region_df.toPandas(), {"Region":"Region","Total_Sales":"Total_Sales", "Total_Profit":"Total_Profit"})
city_pd = standardize_columns(Sales_Profit_City_df.toPandas(), {"City":"City","Total_Sales":"Total_Sales", "Total_Profit":"Total_Profit"})

# Segment Metrics
segment_df = df.groupBy("Segment").agg(
    _sum("Sales").alias("Total_Sales"),
    _sum("Profit").alias("Total_Profit"),
    count("Customer ID").alias("Customer_Count")
)
segment_pd = segment_df.toPandas()
segment_pd = standardize_columns(segment_pd, {
    "Segment":"Segment", "Total_Sales":"Total_Sales", "Total_Profit":"Total_Profit", "Customer_Count":"Customer_Count"
})
segment_pd["Profit_Margin_%"] = (segment_pd["Total_Profit"]/segment_pd["Total_Sales"])*100
segment_pd["Sales_Profit_Ratio"] = (segment_pd["Total_Sales"]/segment_pd["Total_Profit"]).replace([float('inf'), -float('inf')],0)
ratio_pd = segment_pd.sort_values("Sales_Profit_Ratio")

# --- Copying Pdfs to prevent errors ---
rfm_pdf_copy = rfm_pdf.copy()
pred_pdf_copy = pred_pdf.copy()
risk_summary_copy = risk_summary.copy()

# =====================================================
# Month Names
# =====================================================
for df_ in [monthly_sales_pd, profit_month_pd]:
    df_["Order_Month_Name"] = pd.to_datetime(df_["Order_Month"], format="%m", errors="coerce").dt.strftime("%b")
month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
for df_ in [monthly_sales_pd, profit_month_pd]:
    df_["Order_Month_Name"] = pd.Categorical(df_["Order_Month_Name"], categories=month_order, ordered=True)
    df_.sort_values("Order_Month_Name", inplace=True)

# =====================================================
# KPI Metrics
# =====================================================
total_sales = monthly_sales_pd["Total_Sales"].sum()
total_profit = profit_month_pd["Total_Profit"].sum()
total_customers = df.select("Customer ID").distinct().count()

# =====================================================
# Safe plotting functions
# =====================================================
def safe_add_scatter(x, y, row, col, name, color, showlegend=False, **kwargs):
    if len(x) > 0 and len(y) > 0:
        fig.add_trace(go.Scatter(x=x, y=y, mode="lines+markers",
                                 line=dict(color=color, width=3),
                                 name=name, showlegend=showlegend, **kwargs),
                      row=row, col=col)

def safe_add_bar(x, y, row, col, name, color, showlegend=False):
    if len(x) > 0 and len(y) > 0:
        fig.add_trace(go.Bar(x=x, y=y, name=name, marker_color=color, showlegend=showlegend),
                      row=row, col=col)

# =====================================================
# Create Dashboard
# =====================================================
fig = make_subplots(
    rows=8, cols=3,
    specs=[
        [{"type":"indicator"},{"type":"indicator"},{"type":"indicator"}],
        [{"type":"xy"},{"type":"domain"},{"type":"xy"}],
        [{"type":"xy"},{"type":"domain"},{"type":"xy"}],
        [{"type":"xy"},{"type":"xy"},{"type":"xy"}],
        [{"type":"domain"},{"type":"xy"},{"type":"xy"}],
        [{"colspan":3,"rowspan":2,"type":"xy"}, None, None],
        [None, None, None],
        [{"type":"xy"},{"type":"domain"}, {"type":"xy"}] # Changed 'None' to '{"type":"xy"}'
    ],
    subplot_titles=[
        "","","",
        "Monthly Sales","Sales by Category","Sales by SubCategory",
        "Monthly Profit","Profit by Category","Profit by SubCategory",
        "Country Sales vs Profit","Region Sales vs Profit","Top Cities Sales vs Profit",
        "Segment Distribution","Segment Sales vs Profit","Segment Sales Vs Profit Ratio",
        "Customer RFM","Churn Probability Curve","Customer Risk Segmentation",
        "RFM + Churn Segments" # Added title for the new subplot
    ],
    vertical_spacing=0.08
)

# =====================================================
# KPI Indicators
# =====================================================
fig.add_trace(go.Indicator(mode="number", value=total_sales, title={"text":"Total Sales"}), row=1, col=1)
fig.add_trace(go.Indicator(mode="number", value=total_profit, title={"text":"Total Profit"}), row=1, col=2)
fig.add_trace(go.Indicator(mode="number", value=total_customers, title={"text":"Total Customers"}), row=1, col=3)

# =====================================================
# Add Charts
# =====================================================
# Monthly Sales
safe_add_scatter(monthly_sales_pd["Order_Month_Name"], monthly_sales_pd["Total_Sales"], 2,1,"Monthly Sales", SALES_COLOR)

# Sales by Category (Pie)
if not sales_cat.empty and sales_cat["Total_Sales"].sum() > 0:
    fig.add_trace(go.Pie(labels=sales_cat["Category"], values=sales_cat["Total_Sales"], hole=0.3,
                         name="Sales by Category", showlegend=False, marker_colors=OTHER_PIE_COLORS), row=2, col=2)

# Sales by SubCategory (Bar)
safe_add_bar(sales_subcat["SubCategory"], sales_subcat["Total_Sales"], 2,3,"Sales by SubCategory", SALES_COLOR)

# Monthly Profit
safe_add_scatter(profit_month_pd["Order_Month_Name"], profit_month_pd["Total_Profit"], 3,1,"Monthly Profit", PROFIT_COLOR)

# Profit by Category (Pie)
if not profit_cat_pd.empty and profit_cat_pd["Total_Profit"].sum() > 0:
    fig.add_trace(go.Pie(labels=profit_cat_pd["Category"], values=profit_cat_pd["Total_Profit"], hole=0.3,
                         name="Profit by Category", showlegend=True, marker_colors=OTHER_PIE_COLORS), row=3, col=2)

# Profit by SubCategory (Line Chart)
if not profit_subcat_pd.empty and profit_subcat_pd["Total_Profit"].sum() > 0:
    safe_add_scatter(profit_subcat_pd["SubCategory"], profit_subcat_pd["Total_Profit"], 3,3,"Profit by SubCategory", PROFIT_COLOR, showlegend=False)

# Country / Region / City Sales vs Profit (Grouped Bars)
def add_grouped_bar(df, x_col, row, col):
    if not df.empty:
        fig.add_trace(go.Bar(x=df[x_col], y=df["Total_Sales"], name="Sales", marker_color=SALES_COLOR,showlegend=False), row=row, col=col)
        fig.add_trace(go.Bar(x=df[x_col], y=df["Total_Profit"], name="Profit", marker_color=PROFIT_COLOR,showlegend=False), row=row, col=col)

add_grouped_bar(country_pd, "Country", 4,1)
add_grouped_bar(region_pd, "Region", 4,2)
add_grouped_bar(city_pd, "City", 4,3)

# Customer Segment Pie
if not segment_pd.empty and segment_pd["Customer_Count"].sum() > 0:
    seg_colors = [SEGMENT_COLORS.get(seg,"gray") for seg in segment_pd["Segment"]]
    fig.add_trace(go.Pie(
        labels=segment_pd["Segment"],
        values=segment_pd["Customer_Count"],
        hole=0.3,
        name="Segment Distribution",
        marker=dict(colors=seg_colors),
        showlegend=True
    ), row=5, col=1)

# Customer Segment Sales vs Profit (Grouped Bars)
if not segment_pd.empty:
    fig.add_trace(
        go.Bar(
            x=segment_pd["Segment"],
            y=segment_pd["Total_Sales"],
            name="Sales",
            marker_color=SALES_COLOR,
            offsetgroup=0,
            showlegend=True
        ), row=5, col=2
    )
    fig.add_trace(
        go.Bar(
            x=segment_pd["Segment"],
            y=segment_pd["Total_Profit"],
            name="Profit",
            marker_color=PROFIT_COLOR,
            offsetgroup=1,
            showlegend=False
        ), row=5, col=2
    )
# =====================================================
# Customer Segment Sales vs Profit Ratio (Updated)
# =====================================================
if not segment_pd.empty:
    # Sales vs Profit bars
    fig.add_trace(
        go.Bar(
            x=segment_pd["Segment"],
            y=segment_pd["Total_Sales"],
            name="Sales",
            marker_color=SALES_COLOR,
            offsetgroup=0,
            showlegend=False
        ),
        row=5, col=3
    )
    fig.add_trace(
        go.Bar(
            x=segment_pd["Segment"],
            y=segment_pd["Total_Profit"],
            name="Profit",
            marker_color=PROFIT_COLOR,
            offsetgroup=1,
            showlegend=True
        ),
        row=5, col=3
    )
    # Sales to Profit Ratio line
    fig.add_trace(
        go.Scatter(
            x=segment_pd["Segment"],
            y=segment_pd["Sales_Profit_Ratio"],
            mode="lines+markers",
            name="Sales/Profit Ratio",
            line=dict(color=MARGIN_COLOR, width=3),
            yaxis="y2"
        ),
        row=5, col=3
    )

    # Add secondary y-axis for ratio
    fig.update_yaxes(title_text="Sales / Profit Ratio", secondary_y=True, row=5, col=3)
# =====================================================
# Customer RFM (Improved Visualization)
# =====================================================
if not rfm_pdf_copy.empty:

    # Normalize bubble size to avoid giant circles
    max_monetary = rfm_pdf_copy["Monetary"].max()
    size_ref = 2.0 * max_monetary / (40**2)

    fig.add_trace(
        go.Scatter(
            x=rfm_pdf_copy["Frequency"],
            y=rfm_pdf_copy["Monetary"],
            mode="markers",
            marker=dict(
                size=rfm_pdf_copy["Monetary"],
                sizemode="area",
                sizeref=size_ref,
                sizemin=6,
                color=rfm_pdf_copy["Recency"],
                colorscale="Turbo",
                showscale=True,
                colorbar=dict(
                    title="Recency",
                    len=0.20,
                    x=1.05,
                    y=0.25
                ),
                line=dict(width=0.5, color="black")
            ),
            text=rfm_pdf_copy.apply(
                lambda r: f"""
                Customer ID: {r['Customer ID']}
                <br>Recency: {r['Recency']}
                <br>Frequency: {r['Frequency']}
                <br>Monetary: {r['Monetary']}
                """,
                axis=1
            ),
            hoverinfo="text",
            name="Customer RFM",
            showlegend=False
        ),
        row=6,
        col=1
    )

fig.update_xaxes(title_text="Purchase Frequency", row=6, col=1)
fig.update_yaxes(title_text="Customer Monetary Value", row=6, col=1)


# Churn & Risk
if not pred_pdf_copy.empty:
    safe_add_scatter(pred_pdf_copy["Customer_Rank"], pred_pdf_copy["Churn_Prob"], 8,1,"Churn Probability Curve", "orange", showlegend=True)
if not risk_summary_copy.empty:
    fig.add_trace(go.Pie(labels=risk_summary_copy["Risk_Level"], values=risk_summary_copy["Customers"], hole=0.3,
                         name="Customer Risk Segmentation", showlegend=True, marker_colors=OTHER_PIE_COLORS), row=8, col=2)

# =====================================================
# RFM + Churn Segments (Row 8, Column 3) - Fixed Colors
# =====================================================
if not customer_features_pd.empty:
    for seg, color in RFM_SEGMENT_COLORS.items():
        df_seg = customer_features_pd[customer_features_pd["Segment"] == seg]
        if not df_seg.empty:
            fig.add_trace(
                go.Scatter(
                    x=df_seg["Frequency"],
                    y=df_seg["Monetary"],
                    mode="markers",
                    marker=dict(size=10, color=color, line=dict(width=1, color="black")),
                    text=df_seg.apply(
                        lambda r: f"Customer ID: {r['Customer ID']}<br>Recency: {r['Recency']}<br>Monetary: {r['Monetary']}<br>Churn Probability: {r['Churn_Prob']:.2f}", axis=1
                    ),
                    hoverinfo="text",
                    name=seg,
                    showlegend=True
                ),
                row=8, col=3
            )
# =====================================================
# Layout
# =====================================================
fig.update_layout(
    title=dict(
        text="<b>Ecommerce Sales & Profit Analytics Dashboard</b>",
        x=0.5,
        y=0.985,
        font=dict(size=36)
    ),
    height=2900,
    width=1750,
    template="plotly_white",
    margin=dict(t=180, r=180, l=70, b=40),
    barmode='group',
    showlegend=True
)

# Bold subplot titles
for annotation in fig.layout.annotations:
    annotation['text'] = f"<b>{annotation['text']}</b>"
    annotation['y'] += 0.02
    annotation['yanchor'] = 'bottom'

fig.show()

**SEGMENT LEVEL KPI'S**

In [ ]:
# -----------------------------
# 1️⃣ Aggregate Segment-Level KPIs
# -----------------------------
import pandas as pd
from pyspark.sql.functions import sum as _sum

# Assume customer_features_pd has columns: Customer ID, Monetary, Profit, Churn_Prob, Segment
# If Profit is not in customer_features_pd, merge it from df
profit_df = df.groupBy("Customer ID").agg(_sum("Profit").alias("Total_Profit")).toPandas()
customer_features_pd = customer_features_pd.merge(profit_df, on="Customer ID", how="left")

# Aggregate KPIs per Segment
segment_kpis = customer_features_pd.groupby("Segment").agg(
    Avg_Sales=("Monetary", "mean"),
    Avg_Profit=("Total_Profit", "mean"),
    Avg_Churn_Prob=("Churn_Prob", "mean"),
    Customer_Count=("Customer ID", "count")
).reset_index()

# -----------------------------
# 2️⃣ Bubble Chart Visualization
# -----------------------------
import plotly.express as px

fig = px.scatter(
    segment_kpis,
    x="Avg_Sales",
    y="Avg_Profit",
    size="Customer_Count",
    color="Segment",
    hover_name="Segment",
    hover_data={
        "Avg_Sales": ":.2f",
        "Avg_Profit": ":.2f",
        "Avg_Churn_Prob": ":.2f",
        "Customer_Count": True
    },
    title="Segment-Level KPI Analysis: Sales, Profit, Churn Probability",
    size_max=60
)

fig.update_layout(template="plotly_white", xaxis_title="Average Sales", yaxis_title="Average Profit")
fig.show()

 **SEGMENT KPI CALCULATION AND ALL KPI'S IN ONE PLOTILY VISUALIZATION**

In [ ]:
from pyspark.sql.functions import countDistinct, sum as _sum, avg, col, concat_ws, when
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Prepare the Spark DataFrame with Churn_Prob
# We convert the predictions Spark DF to include Churn_Prob as a float column
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType

extract_prob = udf(lambda x: float(x[1]), FloatType())
predictions_with_prob = predictions.withColumn("Churn_Prob", extract_prob(col("probability")))

# 2. Join back to customer_features to get Segments and Monetary values
# We need the 'Segment' column logic applied here
monetary_median_val = customer_features.approxQuantile("Monetary", [0.5], 0.0)[0]

spark_segment_df = customer_features.join(predictions_with_prob.select("Customer ID", "Churn_Prob"), "Customer ID") \
    .withColumn("Value", when(col("Monetary") >= monetary_median_val, "High-Value").otherwise("Low-Value")) \
    .withColumn("Risk", when(col("Churn_Prob") > 0.5, "High-Risk").otherwise("Low-Risk")) \
    .withColumn("Segment", concat_ws(", ", col("Value"), col("Risk")))

# 3. Aggregate KPIs
segment_kpi = spark_segment_df.groupBy("Segment").agg(
    countDistinct("Customer ID").alias("Customer_Count"),
    _sum("Monetary").alias("Total_Revenue"),
    avg("Monetary").alias("Average_Order_Value"),
    avg("Churn_Prob").alias("Avg_Churn_Probability")
)

total_rev = segment_kpi.agg(_sum("Total_Revenue")).collect()[0][0]

segment_kpi = segment_kpi.withColumn(
    "Revenue_Contribution",
    (col("Total_Revenue") / total_rev) * 100
)

segment_kpi_pd = segment_kpi.toPandas()

# 4. Visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Customer Count by Segment",
        "Revenue Contribution",
        "Average Order Value",
        "Segment Performance (Revenue vs Customers)"
    )
)

fig.add_trace(go.Bar(x=segment_kpi_pd["Segment"], y=segment_kpi_pd["Customer_Count"], name="Customer Count"), row=1, col=1)
fig.add_trace(go.Bar(x=segment_kpi_pd["Segment"], y=segment_kpi_pd["Revenue_Contribution"], name="Revenue %", marker_color="orange"), row=1, col=2)
fig.add_trace(go.Bar(x=segment_kpi_pd["Segment"], y=segment_kpi_pd["Average_Order_Value"], name="Avg Order Value", marker_color="green"), row=2, col=1)

fig.add_trace(
    go.Scatter(
        x=segment_kpi_pd["Customer_Count"],
        y=segment_kpi_pd["Total_Revenue"],
        mode="markers+text",
        text=segment_kpi_pd["Segment"],
        marker=dict(
            size=segment_kpi_pd["Average_Order_Value"] / 10,
            color=segment_kpi_pd["Avg_Churn_Probability"],
            colorscale="RdYlGn_r",
            showscale=True
        ),
        name="Segment Performance"
    ),
    row=2, col=2
)

fig.update_layout(height=800, width=1100, title_text="Customer Segment KPI Dashboard", template="plotly_white")
fig.show()


**DASHBOARD**

In [ ]:
# =====================================================
# Imports
# =====================================================
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyspark.sql.functions import sum as _sum, countDistinct, avg, col, concat_ws, when
from pyspark.sql.types import FloatType
from pyspark.sql.functions import udf

# =====================================================
# Segment Colors
# =====================================================
RFM_SEGMENT_COLORS = {
    "High-Value, Low-Risk": "gold",
    "High-Value, High-Risk": "red",
    "Low-Value, Low-Risk": "green",
    "Low-Value, High-Risk": "blue"
}

# =====================================================
# Extract Churn Probability
# =====================================================
extract_prob = udf(lambda x: float(x[1]), FloatType())

predictions_with_prob = predictions.withColumn(
    "Churn_Prob",
    extract_prob(col("probability"))
)

# =====================================================
# Create Value / Risk Segments
# =====================================================
monetary_median_val = customer_features.approxQuantile("Monetary",[0.5],0.0)[0]

spark_segment_df = customer_features.join(
    predictions_with_prob.select("Customer ID","Churn_Prob"),
    "Customer ID"
).withColumn(
    "Value",
    when(col("Monetary") >= monetary_median_val,"High-Value").otherwise("Low-Value")
).withColumn(
    "Risk",
    when(col("Churn_Prob") > 0.5,"High-Risk").otherwise("Low-Risk")
).withColumn(
    "Segment",
    concat_ws(", ",col("Value"),col("Risk"))
)

# =====================================================
# Segment KPIs
# =====================================================
segment_kpi = spark_segment_df.groupBy("Segment").agg(
    countDistinct("Customer ID").alias("Customer_Count"),
    _sum("Monetary").alias("Total_Revenue"),
    avg("Monetary").alias("Average_Order_Value"),
    avg("Churn_Prob").alias("Avg_Churn_Probability")
)

total_rev = segment_kpi.agg(_sum("Total_Revenue")).collect()[0][0]

segment_kpi = segment_kpi.withColumn(
    "Revenue_Contribution",
    (col("Total_Revenue")/total_rev)*100
)

segment_kpi_pd = segment_kpi.toPandas()

# =====================================================
# Profit per Customer
# =====================================================
profit_df = df.groupBy("Customer ID").agg(
    _sum("Profit").alias("Total_Profit")
)

spark_segment_df = spark_segment_df.join(profit_df,"Customer ID","left")

segment_sales_profit = spark_segment_df.groupBy("Segment").agg(
    avg("Monetary").alias("Avg_Sales"),
    avg("Total_Profit").alias("Avg_Profit"),
    avg("Churn_Prob").alias("Avg_Churn"),
    countDistinct("Customer ID").alias("Customer_Count")
)

segment_sales_profit_pd = segment_sales_profit.toPandas()

# =====================================================
# Axis padding
# =====================================================
x_min = segment_sales_profit_pd["Avg_Sales"].min() - 300
x_max = segment_sales_profit_pd["Avg_Sales"].max() + 300

y_min = segment_sales_profit_pd["Avg_Profit"].min() - 100
y_max = segment_sales_profit_pd["Avg_Profit"].max() + 100

# =====================================================
# Create Dashboard
# =====================================================
fig = make_subplots(

    rows=3,
    cols=2,

    specs=[
        [{"type":"xy"},{"type":"xy"}],
        [{"type":"xy"},{"type":"xy"}],
        [{"colspan":2},None]
    ],

    row_heights=[0.32,0.32,0.45],

    subplot_titles=(
        "Customer Count by Segment",
        "Revenue Contribution (%)",
        "Average Order Value",
        "Segment Performance (Revenue vs Customers)",
        "Customer Value vs Churn Risk Matrix"
    ),

    vertical_spacing=0.20
)

# =====================================================
# Chart 1 Customer Count
# =====================================================
for seg,color in RFM_SEGMENT_COLORS.items():

    data = segment_kpi_pd[segment_kpi_pd["Segment"]==seg]

    if not data.empty:

        fig.add_trace(
            go.Bar(
                x=data["Segment"],
                y=data["Customer_Count"],
                marker_color=color,
                showlegend=False
            ),
            row=1,col=1
        )

# =====================================================
# Chart 2 Revenue Contribution
# =====================================================
for seg,color in RFM_SEGMENT_COLORS.items():

    data = segment_kpi_pd[segment_kpi_pd["Segment"]==seg]

    if not data.empty:

        fig.add_trace(
            go.Bar(
                x=data["Segment"],
                y=data["Revenue_Contribution"],
                marker_color=color,
                showlegend=False
            ),
            row=1,col=2
        )

# =====================================================
# Chart 3 Average Order Value
# =====================================================
for seg,color in RFM_SEGMENT_COLORS.items():

    data = segment_kpi_pd[segment_kpi_pd["Segment"]==seg]

    if not data.empty:

        fig.add_trace(
            go.Bar(
                x=data["Segment"],
                y=data["Average_Order_Value"],
                marker_color=color,
                showlegend=False
            ),
            row=2,col=1
        )

# =====================================================
# Chart 4 Revenue vs Customers
# =====================================================
for seg,color in RFM_SEGMENT_COLORS.items():

    data = segment_kpi_pd[segment_kpi_pd["Segment"]==seg]

    if not data.empty:

        fig.add_trace(
            go.Scatter(
                x=data["Customer_Count"],
                y=data["Total_Revenue"],
                mode="markers",
                marker=dict(
                    size=data["Average_Order_Value"]/40,
                    color=color
                ),
                showlegend=False
            ),
            row=2,col=2
        )

# =====================================================
# Chart 5 Customer Value vs Churn Risk Matrix
# =====================================================
size_scale = 5

for seg, color in RFM_SEGMENT_COLORS.items():

    data = segment_sales_profit_pd[
        segment_sales_profit_pd["Segment"] == seg
    ]

    if not data.empty:

        fig.add_trace(
            go.Scatter(
                x=data["Avg_Sales"],
                y=data["Avg_Profit"],
                mode="markers",
                name=seg,
                marker=dict(
                    size=data["Customer_Count"] * size_scale,
                    color=color,
                    sizemode="area",
                    line=dict(width=1, color="black"),
                    opacity=0.85
                )
            ),
            row=3, col=1
        )

# =====================================================
# Axis Range
# =====================================================
fig.update_xaxes(
    title_text="Average Sales",
    range=[x_min,x_max],
    row=3,col=1
)

fig.update_yaxes(
    title_text="Average Profit",
    range=[y_min,y_max],
    row=3,col=1
)

# =====================================================
# Layout
# =====================================================
fig.update_layout(

    height=1500,
    width=1200,

    title=dict(
        text="📊 Customer Segment KPI & Performance Dashboard",
        x=0.5,
        y=0.96,
        xanchor="center",
        font=dict(size=36,family="Arial Black")
    ),

    margin=dict(
        t=220,
        b=80,
        l=80,
        r=80
    ),

    template="plotly_white",

    legend=dict(
        title="Customer Segments",
        font=dict(size=14)
    )
)

# Bold subplot titles
for ann in fig["layout"]["annotations"]:
    ann["font"]=dict(size=18,family="Arial Black")

fig.show()

**CUSTOMER LIFETIME VALUE ANALYSIS**

In [ ]:
# ============================================================
# 1️⃣ Import Libraries
# ============================================================
from pyspark.sql.functions import col, when, concat_ws, avg
import plotly.express as px

# ============================================================
# 2️⃣ Ensure Segment column is in Spark DataFrame
# ============================================================
# Re-calculating Segment logic in Spark to fix the missing column error
monetary_med = customer_features.approxQuantile("Monetary", [0.5], 0.0)[0]

# Add 'churn' column to customer_features before using it
customer_features = customer_features.withColumn(
    "churn",
    when(col("Recency") > 250, 1).otherwise(0) # Using 250 days as churn threshold, consistent with previous churn model
)

# We join the churn information if needed, but assuming customer_features
# already has 'churn' or 'Recency' from previous steps
customer_features = customer_features.withColumn(
    "Value",
    when(col("Monetary") >= monetary_med, "High-Value").otherwise("Low-Value")
).withColumn(
    "Risk",
    when(col("churn") == 1, "High-Risk").otherwise("Low-Risk")
).withColumn(
    "Segment",
    concat_ws(", ", col("Value"), col("Risk"))
)

# ============================================================
# 3️⃣ Calculate Customer Metrics
# ============================================================

# Average Order Value
customer_features = customer_features.withColumn(
    "Avg_Order_Value",
    col("Monetary") / col("Frequency")
)

# Estimate Customer Lifespan (based on Recency - avoid division by zero)
customer_features = customer_features.withColumn(
    "Customer_Lifespan",
    when(col("Recency") > 0, 365 / col("Recency")).otherwise(365)
)

# Customer Lifetime Value
customer_features = customer_features.withColumn(
    "CLV",
    col("Avg_Order_Value") * col("Frequency") * col("Customer_Lifespan")
)

# Preview results
customer_features.select(
    "Customer ID",
    "Recency",
    "Frequency",
    "Monetary",
    "Avg_Order_Value",
    "CLV",
    "Segment"
).show(10)

# ============================================================
# 4️⃣ Convert to Pandas for Visualization
# ============================================================

clv_pd = customer_features.select(
    "Customer ID",
    "Frequency",
    "Monetary",
    "CLV",
    "Segment"
).toPandas()

# ============================================================
# 5️⃣ CLV Distribution
# ============================================================

fig1 = px.histogram(
    clv_pd,
    x="CLV",
    nbins=40,
    title="Customer Lifetime Value Distribution"
)

fig1.update_layout(template="plotly_white")
fig1.show()

# ============================================================
# 6️⃣ CLV vs Purchase Frequency (Bubble Chart)
# ============================================================

fig2 = px.scatter(
    clv_pd,
    x="Frequency",
    y="CLV",
    size="Monetary",
    color="CLV",
    hover_data=["Customer ID"],
    title="Customer Lifetime Value vs Purchase Frequency"
)

fig2.update_layout(template="plotly_white")
fig2.show()

# ============================================================
# 7️⃣ CLV by Customer Segment
# ============================================================

clv_segment_pd = clv_pd.groupby("Segment")["CLV"].mean().reset_index()

fig3 = px.bar(
    clv_segment_pd,
    x="Segment",
    y="CLV",
    color="Segment",
    title="Average Customer Lifetime Value by Segment"
)

fig3.update_layout(template="plotly_white")
fig3.show()

**DASHBOARD**

In [ ]:
# =====================================================
# Imports
# =====================================================
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyspark.sql.functions import sum as _sum, countDistinct, avg, col, concat_ws, when
from pyspark.sql.types import FloatType
from pyspark.sql.functions import udf


# =====================================================
# Segment Colors
# =====================================================
RFM_SEGMENT_COLORS = {
    "High-Value, Low-Risk": "gold",
    "High-Value, High-Risk": "red",
    "Low-Value, Low-Risk": "green",
    "Low-Value, High-Risk": "blue"
}

# =====================================================
# Extract Churn Probability
# =====================================================
extract_prob = udf(lambda x: float(x[1]), FloatType())

predictions_with_prob = predictions.withColumn(
    "Churn_Prob",
    extract_prob(col("probability"))
)

# =====================================================
# Create Segments
# =====================================================
monetary_median_val = customer_features.approxQuantile("Monetary",[0.5],0.0)[0]

spark_segment_df = customer_features.join(
    predictions_with_prob.select("Customer ID","Churn_Prob"),
    "Customer ID"
).withColumn(
    "Value",
    when(col("Monetary") >= monetary_median_val,"High-Value").otherwise("Low-Value")
).withColumn(
    "Risk",
    when(col("Churn_Prob") > 0.5,"High-Risk").otherwise("Low-Risk")
).withColumn(
    "Segment",
    concat_ws(", ",col("Value"),col("Risk"))
)

# =====================================================
# CLV Calculation
# =====================================================

spark_segment_df = spark_segment_df.withColumn(
    "Avg_Order_Value",
    col("Monetary") / col("Frequency")
)

spark_segment_df = spark_segment_df.withColumn(
    "Customer_Lifespan",
    when(col("Recency") > 0, 365/col("Recency")).otherwise(365)
)

spark_segment_df = spark_segment_df.withColumn(
    "CLV",
    col("Avg_Order_Value") * col("Frequency") * col("Customer_Lifespan")
)

# =====================================================
# Segment KPIs
# =====================================================

segment_kpi = spark_segment_df.groupBy("Segment").agg(
    countDistinct("Customer ID").alias("Customer_Count"),
    _sum("Monetary").alias("Total_Revenue"),
    avg("Monetary").alias("Average_Order_Value"),
    avg("CLV").alias("Avg_CLV")
)

total_rev = segment_kpi.agg(_sum("Total_Revenue")).collect()[0][0]

segment_kpi = segment_kpi.withColumn(
    "Revenue_Contribution",
    (col("Total_Revenue")/total_rev)*100
)

segment_kpi_pd = segment_kpi.toPandas()

# =====================================================
# Profit Analysis
# =====================================================

profit_df = df.groupBy("Customer ID").agg(
    _sum("Profit").alias("Total_Profit")
)

spark_segment_df = spark_segment_df.join(profit_df,"Customer ID","left")

segment_sales_profit = spark_segment_df.groupBy("Segment").agg(
    avg("Monetary").alias("Avg_Sales"),
    avg("Total_Profit").alias("Avg_Profit"),
    countDistinct("Customer ID").alias("Customer_Count")
)

segment_sales_profit_pd = segment_sales_profit.toPandas()

# =====================================================
# CLV Data
# =====================================================

clv_pd = spark_segment_df.select(
    "Customer ID",
    "Frequency",
    "Monetary",
    "CLV",
    "Segment"
).toPandas()

clv_segment_pd = clv_pd.groupby("Segment")["CLV"].mean().reset_index()

# =====================================================
# Dashboard Layout
# =====================================================

fig = make_subplots(

rows=5,
cols=2,

specs=[
[{"type":"xy"},{"type":"xy"}],
[{"type":"xy"},{"type":"xy"}],
[{"colspan":2},None],
[{"type":"xy"},{"type":"xy"}],
[{"colspan":2},None]
],

subplot_titles=(

"Customer Count by Segment",
"Revenue Contribution (%)",
"Average Order Value",
"Segment Performance (Revenue vs Customers)",
"Customer Value vs Churn Risk Matrix",
"Average CLV by Segment",
"CLV Distribution",
"CLV vs Purchase Frequency"

),

vertical_spacing=0.10
)

# =====================================================
# Chart 1 Customer Count
# =====================================================

for seg,color in RFM_SEGMENT_COLORS.items():

    data = segment_kpi_pd[segment_kpi_pd["Segment"]==seg]

    if not data.empty:

        fig.add_trace(
            go.Bar(
                x=data["Segment"],
                y=data["Customer_Count"],
                marker_color=color,
                showlegend=False
            ),
            row=1,col=1
        )

# =====================================================
# Chart 2 Revenue Contribution
# =====================================================

for seg,color in RFM_SEGMENT_COLORS.items():

    data = segment_kpi_pd[segment_kpi_pd["Segment"]==seg]

    if not data.empty:

        fig.add_trace(
            go.Bar(
                x=data["Segment"],
                y=data["Revenue_Contribution"],
                marker_color=color,
                showlegend=False
            ),
            row=1,col=2
        )

# =====================================================
# Chart 3 Average Order Value
# =====================================================

for seg,color in RFM_SEGMENT_COLORS.items():

    data = segment_kpi_pd[segment_kpi_pd["Segment"]==seg]

    if not data.empty:

        fig.add_trace(
            go.Bar(
                x=data["Segment"],
                y=data["Average_Order_Value"],
                marker_color=color,
                showlegend=False
            ),
            row=2,col=1
        )

# =====================================================
# Chart 4 Segment Performance
# =====================================================

for seg,color in RFM_SEGMENT_COLORS.items():

    data = segment_kpi_pd[segment_kpi_pd["Segment"]==seg]

    if not data.empty:

        fig.add_trace(
            go.Scatter(
                x=data["Customer_Count"],
                y=data["Total_Revenue"],
                mode="markers",
                marker=dict(
                    size=data["Average_Order_Value"]/50,
                    color=color,
                    opacity=0.8
                ),
                showlegend=False
            ),
            row=2,col=2
        )

# =====================================================
# Chart 5 Customer Value vs Churn Risk Matrix
# =====================================================

size_scale = 0.4   # reduced bubble size

for seg,color in RFM_SEGMENT_COLORS.items():

    data = segment_sales_profit_pd[
        segment_sales_profit_pd["Segment"]==seg
    ]

    if not data.empty:

        fig.add_trace(
            go.Scatter(
                x=data["Avg_Sales"],
                y=data["Avg_Profit"],
                mode="markers",
                name=seg,
                marker=dict(
                    size=data["Customer_Count"]*size_scale,
                    color=color,
                    line=dict(width=1,color="black"),
                    opacity=0.8
                )
            ),
            row=3,col=1
        )

# =====================================================
# Chart 6 CLV by Segment
# =====================================================

fig.add_trace(
    go.Bar(
        x=clv_segment_pd["Segment"],
        y=clv_segment_pd["CLV"],
        marker_color="purple"
    ),
    row=4,col=1
)

# =====================================================
# Chart 7 CLV Distribution
# =====================================================

fig.add_trace(
    go.Histogram(
        x=clv_pd["CLV"],
        nbinsx=40,
        marker_color="skyblue"
    ),
    row=4,col=2
)

# =====================================================
# Chart 8 CLV vs Purchase Frequency
# =====================================================

fig.add_trace(
    go.Scatter(
        x=clv_pd["Frequency"],
        y=clv_pd["CLV"],
        mode="markers",
        marker=dict(
            size=clv_pd["Monetary"]/300,   # reduced bubble size
            color=clv_pd["CLV"],
            colorscale="Viridis",
            opacity=0.7,
            colorbar=dict(
                title="CLV",
                thickness=12,
                len=0.2,
                y=0.0990
            )
        )
    ),
    row=5,col=1
)

# =====================================================
# Layout
# =====================================================

fig.update_layout(

height=2000,
width=1300,

title=dict(
text="📊 Customer Segmentation, Churn Risk & CLV Analytics Dashboard",
x=0.5,
font=dict(size=34,family="Arial Black")
),

template="plotly_white"
)

fig.show()


**Product Affinity / Market Basket Analysis**

In [ ]:
# ============================================================
# 1️⃣ Install Required Libraries
# ============================================================
!pip install mlxtend -q

# ============================================================
# 2️⃣ Imports and Warnings
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import plotly.express as px
from mlxtend.frequent_patterns import apriori, association_rules

# ============================================================
# 3️⃣ Convert Spark DF → Pandas
# ============================================================
orders_pd = df.select("Order ID", "Product Name").toPandas()

# ============================================================
# 4️⃣ Optional: Reduce Dataset Size
# ============================================================
top_n = 50
top_products = orders_pd["Product Name"].value_counts().nlargest(top_n).index
orders_pd = orders_pd[orders_pd["Product Name"].isin(top_products)].drop_duplicates()

# ============================================================
# 5️⃣ Basket / Pivot Table
# ============================================================
basket = orders_pd.pivot_table(
    index="Order ID",
    columns="Product Name",
    aggfunc="size",
    fill_value=0
)

basket_encoded = (basket > 0)  # Convert counts → 0/1

# ============================================================
# 6️⃣ Apriori Algorithm
# ============================================================
frequent_itemsets = apriori(
    basket_encoded,
    min_support=0.002,
    use_colnames=True
)

# ============================================================
# 7️⃣ Association Rules
# ============================================================
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1
).sort_values("lift", ascending=False)

print(f"Total rules found: {len(rules)}")

# ============================================================
# 8️⃣ Prepare Columns for Visualization
# ============================================================
if not rules.empty:

    # Convert frozensets to strings
    rules["antecedents_str"] = rules["antecedents"].apply(lambda x: ", ".join(list(x)))
    rules["consequents_str"] = rules["consequents"].apply(lambda x: ", ".join(list(x)))

    # Limit top 200 rules for plotting
    vis_rules = rules.head(200).copy()

    # ============================================================
    # Scatter Chart: Support vs Confidence
    # ============================================================
    max_bubble_size = 25
    sizeref1 = 2.0 * vis_rules["lift"].max() / (max_bubble_size**2)

    fig1 = px.scatter(
        vis_rules,
        x="support",
        y="confidence",
        size="lift",
        color="lift",
        hover_data=["antecedents_str", "consequents_str"],
        color_continuous_scale="Turbo",
        title="Market Basket Analysis: Support vs Confidence"
    )
    fig1.update_traces(
        marker=dict(
            sizemode="area",
            sizeref=sizeref1,
            line=dict(width=1, color="black"),
            opacity=0.75
        )
    )
    fig1.update_layout(
        template="plotly_white",
        height=650,
        coloraxis_colorbar=dict(title="Lift", thickness=15, len=0.7)
    )
    fig1.show()

    # ============================================================
    # Top Product Affinity Rules (Bar Chart)
    # ============================================================
    top_rules = rules.head(15).copy()

    fig2 = px.bar(
        top_rules,
        x="antecedents_str",
        y="lift",
        color="confidence",
        color_continuous_scale="Viridis",
        hover_data=["consequents_str", "support"],
        title="Top Product Affinity Rules (Highest Lift)"
    )
    fig2.update_layout(
        template="plotly_white",
        height=650,
        xaxis_tickangle=-45
    )
    fig2.show()

    # ============================================================
    # Scatter Chart: Lift vs Confidence
    # ============================================================
    max_bubble_size2 = 25
    sizeref2 = 2.0 * vis_rules["support"].max() / (max_bubble_size2**2)

    fig3 = px.scatter(
        vis_rules,
        x="lift",
        y="confidence",
        size="support",
        color="support",
        hover_data=["antecedents_str", "consequents_str"],
        color_continuous_scale="Plasma",
        title="Lift vs Confidence for Association Rules"
    )
    fig3.update_traces(
        marker=dict(
            sizemode="area",
            sizeref=sizeref2,
            line=dict(width=1, color="black"),
            opacity=0.75
        )
    )
    fig3.update_layout(
        template="plotly_white",
        height=650,
        coloraxis_colorbar=dict(title="Support", thickness=15, len=0.7)
    )
    fig3.show()

else:
    print("No association rules found. Try lowering min_support to 0.001")

## **Customer Purchase Pattern Analysis**

In [ ]:
# ============================================================
# 1️⃣ Imports
# ============================================================
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import pandas as pd
import plotly.express as px

# Ensure SparkSession and df are available if the state was lost
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, year, month, date_format

# Initialize SparkSession (gets existing or creates new one)
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("CustomerPurchasePattern") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")\
    .getOrCreate()

# Re-load df with inferSchema if it's not defined or lost
# This uses the same loading parameters as the initial df creation in JckuJmaK9E7H
# and subsequent date conversions in et82o-mMD5xx and eHWo0D1FQ8LJ
if 'df' not in locals():
    df = spark.read.csv("/content/drive/MyDrive/sample.csv",
                        header=True,
                        inferSchema=True,
                        encoding='ISO-8859-1')

    df = df.withColumn(
        "Order Date",
        F.to_date(F.col("Order Date"), "MM/dd/yyyy")
    )

    df = df.withColumn(
        "Ship Date",
        F.to_date(F.col("Ship Date"), "MM/dd/yyyy")
    )

    df = df.withColumn("Order Year", year(col("Order Date"))) \
           .withColumn("Order Month", month(col("Order Date"))) \
           .withColumn("Day Of Week", date_format(col("Order Date"), "EEEE"))


# ============================================================
# 2️⃣ Sample Transaction Data (replace with your CSV or Spark DF)
# ============================================================
# Columns: Customer ID, Order ID, Product Name, Order Date
orders_pd = df.select("Order ID", "Product Name","Customer ID","Order Date").toPandas()

# Convert 'Order Date' to datetime type in Pandas
orders_pd['Order Date'] = pd.to_datetime(orders_pd['Order Date'])

# ============================================================
# 3️⃣ Compute Customer Metrics
# ============================================================

# 3a: Average basket size (items per order)
basket_size = orders_pd.groupby("Order ID")["Product Name"].count().reset_index()
basket_size.rename(columns={"Product Name":"Basket_Size"}, inplace=True)
avg_basket_size = basket_size["Basket_Size"].mean()
print(f"Average Basket Size: {avg_basket_size:.2f} items per order")

# 3b: Repeat purchase rate (% customers with >1 order)
customer_orders = orders_pd.groupby("Customer ID")["Order ID"].nunique().reset_index()
customer_orders.rename(columns={"Order ID":"Num_Orders"}, inplace=True)
repeat_purchase_rate = (customer_orders["Num_Orders"] > 1).mean() * 100
print(f"Repeat Purchase Rate: {repeat_purchase_rate:.2f}%")

# 3c: Average order interval (days between purchases)
orders_sorted = orders_pd.sort_values(["Customer ID","Order Date"])
orders_sorted["Prev_Order_Date"] = orders_sorted.groupby("Customer ID")["Order Date"].shift(1)
orders_sorted["Order_Interval"] = (orders_sorted["Order Date"] - orders_sorted["Prev_Order_Date"]).dt.days

avg_order_interval = orders_sorted["Order_Interval"].mean()
print(f"Average Order Interval: {avg_order_interval:.2f} days")

# ============================================================
# 4️⃣ Visualizations
# ============================================================

# 4a: Repeat Purchase Distribution
fig1 = px.histogram(customer_orders, x="Num_Orders", nbins=10,
                    title="Customer Repeat Purchase Distribution")
fig1.update_layout(template="plotly_white", xaxis_title="Number of Orders", yaxis_title="Number of Customers")
fig1.show()

# 4b: Basket Size Distribution
fig2 = px.histogram(basket_size, x="Basket_Size", nbins=10,
                    title="Order Basket Size Distribution")
fig2.update_layout(template="plotly_white", xaxis_title="Items per Order", yaxis_title="Number of Orders")
fig2.show()

# 4c: Average Order Interval per Customer
intervals = orders_sorted.groupby("Customer ID")["Order_Interval"].mean().reset_index()
fig3 = px.bar(intervals, x="Customer ID", y="Order_Interval",
              title="Average Order Interval per Customer (Days)")
fig3.update_layout(template="plotly_white", yaxis_title="Days")
fig3.show()


**DASHBOARD**

In [ ]:
# ============================================================
# 1️⃣ Install Required Library
# ============================================================

!pip install mlxtend -q


# ============================================================
# 2️⃣ Imports
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import plotly.express as px

from mlxtend.frequent_patterns import apriori, association_rules

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, year, month, date_format


# ============================================================
# 3️⃣ Start Spark
# ============================================================

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("CustomerPurchasePattern") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()


# ============================================================
# 4️⃣ Load Dataset
# ============================================================

df = spark.read.csv(
    "/content/drive/MyDrive/sample.csv",
    header=True,
    inferSchema=True,
    encoding='ISO-8859-1'
)

df = df.withColumn("Order Date", F.to_date(F.col("Order Date"), "MM/dd/yyyy")) \
       .withColumn("Ship Date", F.to_date(F.col("Ship Date"), "MM/dd/yyyy")) \
       .withColumn("Order Year", year(col("Order Date"))) \
       .withColumn("Order Month", month(col("Order Date"))) \
       .withColumn("Day Of Week", date_format(col("Order Date"), "EEEE"))


# ============================================================
# 5️⃣ Convert Spark Data → Pandas
# ============================================================

orders_pd = df.select(
    "Order ID",
    "Product Name",
    "Sub-Category",
    "Customer ID",
    "Order Date"
).toPandas()

orders_pd["Order Date"] = pd.to_datetime(orders_pd["Order Date"])


# ============================================================
# 6️⃣ Customer Purchase Metrics
# ============================================================

# Basket Size
basket_size = orders_pd.groupby("Order ID")["Product Name"].count().reset_index()
basket_size.rename(columns={"Product Name":"Basket_Size"}, inplace=True)

avg_basket = basket_size["Basket_Size"].mean()
print("Average Basket Size:", round(avg_basket,2))


# Repeat Purchase Rate
customer_orders = orders_pd.groupby("Customer ID")["Order ID"].nunique().reset_index()
customer_orders.rename(columns={"Order ID":"Num_Orders"}, inplace=True)

repeat_rate = (customer_orders["Num_Orders"] > 1).mean() * 100
print("Repeat Purchase Rate:", round(repeat_rate,2), "%")


# Average Order Interval
orders_sorted = orders_pd.sort_values(["Customer ID","Order Date"])

orders_sorted["Prev_Order"] = orders_sorted.groupby("Customer ID")["Order Date"].shift(1)

orders_sorted["Order_Interval"] = (
        orders_sorted["Order Date"] -
        orders_sorted["Prev_Order"]
).dt.days

intervals = orders_sorted.groupby("Customer ID")["Order_Interval"].mean().reset_index()


# ============================================================
# 7️⃣ Market Basket Analysis
# ============================================================

orders_subset = orders_pd[["Order ID","Sub-Category"]].drop_duplicates()

basket = orders_subset.pivot_table(
    index="Order ID",
    columns="Sub-Category",
    aggfunc=len,
    fill_value=0
)

basket_encoded = basket.applymap(lambda x: 1 if x>0 else 0)


# Apriori Algorithm
frequent_itemsets = apriori(
    basket_encoded,
    min_support=0.001,
    use_colnames=True
)


# Association Rules
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1
)


# Filter Strong Rules
rules = rules[
    (rules["confidence"] > 0.05) &
    (rules["lift"] > 1.1)
]

rules = rules.sort_values("lift", ascending=False)

rules["antecedents_str"] = rules["antecedents"].apply(lambda x: ", ".join(list(x)))
rules["consequents_str"] = rules["consequents"].apply(lambda x: ", ".join(list(x)))

print("Total Rules Generated:", len(rules))

vis_rules = rules.head(200)


# ============================================================
# 8️⃣ Customer Behaviour Visualizations
# ============================================================

# Repeat Purchase Distribution
fig1 = px.histogram(
    customer_orders,
    x="Num_Orders",
    nbins=10,
    title="Customer Repeat Purchase Distribution"
)

fig1.update_layout(template="plotly_white")

fig1.show()


# Basket Size Distribution
fig2 = px.histogram(
    basket_size,
    x="Basket_Size",
    nbins=10,
    title="Order Basket Size Distribution"
)

fig2.update_layout(template="plotly_white")

fig2.show()


# Customer Order Interval
interval_sample = intervals.sort_values("Order_Interval", ascending=False).head(50)

fig3 = px.bar(
    interval_sample,
    x="Customer ID",
    y="Order_Interval",
    title="Top Customers by Average Order Interval"
)

fig3.update_layout(template="plotly_white")

fig3.show()


# ============================================================
# 9️⃣ Market Basket Visualizations
# ============================================================

if not vis_rules.empty:

    # Support vs Confidence
    fig4 = px.scatter(
        vis_rules,
        x="support",
        y="confidence",
        size="lift",
        color="lift",
        hover_data=["antecedents_str","consequents_str"],
        title="Market Basket Analysis: Support vs Confidence"
    )

    fig4.update_layout(template="plotly_white")

    fig4.show()


    # Top Product Affinity Rules
    top_rules = vis_rules.head(15)

    fig5 = px.bar(
        top_rules,
        x="antecedents_str",
        y="lift",
        color="confidence",
        hover_data=["consequents_str","support"],
        title="Top Product Affinity Rules"
    )

    fig5.update_layout(template="plotly_white")

    fig5.show()


    # Lift vs Confidence
    fig6 = px.scatter(
        vis_rules,
        x="lift",
        y="confidence",
        size="support",
        color="support",
        hover_data=["antecedents_str","consequents_str"],
        title="Lift vs Confidence for Association Rules"
    )

    fig6.update_layout(template="plotly_white")

    fig6.show()

else:
    print("No association rules found. Try lowering min_support.")

**DASHBOARD**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px


# =====================================================
# DASHBOARD LAYOUT
# =====================================================

fig = make_subplots(

rows=8,
cols=2,

specs=[
[{"type":"xy"},{"type":"xy"}],
[{"type":"xy"},{"type":"xy"}],
[{"colspan":2},None],
[{"type":"xy"},{"type":"xy"}],
[{"colspan":2},None],
[{"type":"xy"},{"type":"xy"}],
[{"type":"xy"},{"type":"xy"}],
[{"type":"xy"},{"type":"xy"}]
],

subplot_titles=(

"<b>Customer Count by Segment</b>",
"<b>Revenue Contribution</b>",

"<b>Average Order Value</b>",
"<b>Segment Performance</b>",

"<b>Customer Value vs Churn Risk Matrix</b>",

"<b>Average CLV by Segment</b>",
"<b>CLV Distribution</b>",

"<b>CLV vs Purchase Frequency</b>",

"<b>Repeat Purchase Distribution</b>",
"<b>Basket Size Distribution</b>",

"<b>Customer Order Interval</b>",

"<b>Market Basket: Support vs Confidence</b>",
"<b>Top Product Affinity Rules</b>",

"<b>Lift vs Confidence</b>",
""

),

vertical_spacing=0.09
)


# =====================================================
# SEGMENT COLORS
# =====================================================

RFM_SEGMENT_COLORS={
"High-Value, Low-Risk":"gold",
"High-Value, High-Risk":"red",
"Low-Value, Low-Risk":"green",
"Low-Value, High-Risk":"blue"
}


# =====================================================
# CUSTOMER SEGMENT CHARTS
# =====================================================

for seg,color in RFM_SEGMENT_COLORS.items():

    data=segment_kpi_pd[segment_kpi_pd["Segment"]==seg]

    if not data.empty:

        fig.add_trace(go.Bar(
            x=data["Segment"],
            y=data["Customer_Count"],
            marker_color=color,
            name=seg
        ),row=1,col=1)

        fig.add_trace(go.Bar(
            x=data["Segment"],
            y=data["Revenue_Contribution"],
            marker_color=color,
            showlegend=False
        ),row=1,col=2)

        fig.add_trace(go.Bar(
            x=data["Segment"],
            y=data["Average_Order_Value"],
            marker_color=color,
            showlegend=False
        ),row=2,col=1)

        fig.add_trace(go.Scatter(
            x=data["Customer_Count"],
            y=data["Total_Revenue"],
            mode="markers",
            marker=dict(
                size=data["Average_Order_Value"]/120,
                color=color
            ),
            showlegend=False
        ),row=2,col=2)


# =====================================================
# CHURN MATRIX
# =====================================================

for seg,color in RFM_SEGMENT_COLORS.items():

    data=segment_sales_profit_pd[
        segment_sales_profit_pd["Segment"]==seg
    ]

    if not data.empty:

        fig.add_trace(go.Scatter(
            x=data["Avg_Sales"],
            y=data["Avg_Profit"],
            mode="markers",
            marker=dict(
                size=data["Customer_Count"]/8,
                color=color,
                line=dict(width=1,color="black")
            ),
            name=seg
        ),row=3,col=1)


# =====================================================
# CLV CHARTS
# =====================================================

fig.add_trace(go.Bar(
    x=clv_segment_pd["Segment"],
    y=clv_segment_pd["CLV"],
    marker_color="purple",
    name="CLV"
),row=4,col=1)

fig.add_trace(go.Histogram(
    x=clv_pd["CLV"],
    nbinsx=40,
    marker_color="skyblue",
    name="CLV Distribution"
),row=4,col=2)


# CLV Scatter
fig.add_trace(go.Scatter(

    x=clv_pd["Frequency"],
    y=clv_pd["CLV"],
    mode="markers",

    marker=dict(
        size=clv_pd["Monetary"]/600,
        color=clv_pd["CLV"],
        colorscale="Viridis",
        showscale=True,

        colorbar=dict(
            title="CLV",
            len=0.11,
            y=0.46,
            thickness=10
        )
    ),

    name="CLV vs Frequency"

),row=5,col=1)


# =====================================================
# CUSTOMER BEHAVIOUR
# =====================================================

repeat_fig=px.histogram(customer_orders,x="Num_Orders")
for t in repeat_fig.data:
    fig.add_trace(t,row=6,col=1)

basket_fig=px.histogram(basket_size,x="Basket_Size")
for t in basket_fig.data:
    fig.add_trace(t,row=6,col=2)

interval_fig=px.bar(interval_sample,x="Customer ID",y="Order_Interval")
for t in interval_fig.data:
    fig.add_trace(t,row=7,col=1)


# =====================================================
# MARKET BASKET ANALYSIS
# =====================================================

fig.add_trace(go.Scatter(

    x=vis_rules["support"],
    y=vis_rules["confidence"],
    mode="markers",

    marker=dict(
        size=vis_rules["lift"]*4,
        color=vis_rules["lift"],
        colorscale="Turbo",
        showscale=True,

        colorbar=dict(
            title="Lift",
            len=0.10,
            y=0.18,
            thickness=10
        )
    ),

    text=vis_rules["antecedents_str"],
    name="Association Rules"

),row=7,col=2)


# =====================================================
# TOP PRODUCT RULES
# =====================================================

top_rules=vis_rules.head(15)

fig.add_trace(go.Bar(
    x=top_rules["antecedents_str"],
    y=top_rules["lift"],

    marker=dict(
        color=top_rules["confidence"],
        colorscale="Blues",
        showscale=True,

        colorbar=dict(
            title="Confidence",
            len=0.11,
            y=0.59,
            thickness=10
        )
    ),

    name="Top Rules"

),row=8,col=1)

# =====================================================
# LIFT VS CONFIDENCE
# =====================================================
fig.add_trace(go.Scatter(

    x=vis_rules["lift"],
    y=vis_rules["confidence"],
    mode="markers",

    marker=dict(
        size=vis_rules["support"]*7000,
        color=vis_rules["support"],
        colorscale="Plasma",
        showscale=True,

        colorbar=dict(
            title="Support",
            len=0.10,
            y=0.045,
            thickness=10
        )
    ),

    text=vis_rules["antecedents_str"],
    name="Lift vs Confidence"

),row=8,col=2)


# =====================================================
# AXIS LABELS
# =====================================================

fig.update_xaxes(title_text="Segment", row=1, col=1)
fig.update_yaxes(title_text="Customer Count", row=1, col=1)

fig.update_xaxes(title_text="Segment", row=1, col=2)
fig.update_yaxes(title_text="Revenue Contribution", row=1, col=2)

fig.update_xaxes(title_text="Segment", row=2, col=1)
fig.update_yaxes(title_text="Average Order Value", row=2, col=1)

fig.update_xaxes(title_text="Customer Count", row=2, col=2)
fig.update_yaxes(title_text="Total Revenue", row=2, col=2)

fig.update_xaxes(title_text="Avg Sales", row=3, col=1)
fig.update_yaxes(title_text="Avg Profit", row=3, col=1)

fig.update_xaxes(title_text="Segment", row=4, col=1)
fig.update_yaxes(title_text="Customer Lifetime Value", row=4, col=1)

fig.update_xaxes(title_text="CLV", row=4, col=2)
fig.update_yaxes(title_text="Frequency", row=4, col=2)

fig.update_xaxes(title_text="Purchase Frequency", row=5, col=1)
fig.update_yaxes(title_text="CLV", row=5, col=1)

# Row 6 labels
fig.update_xaxes(title_text="Number of Orders", row=6, col=1)
fig.update_yaxes(title_text="Customer Count", row=6, col=1)

fig.update_xaxes(title_text="Basket Size", row=6, col=2)
fig.update_yaxes(title_text="Frequency", row=6, col=2)

# Row 7 labels
fig.update_xaxes(title_text="Customer ID", row=7, col=1)
fig.update_yaxes(title_text="Order Interval (Days)", row=7, col=1)

fig.update_xaxes(title_text="Support", row=7, col=2)
fig.update_yaxes(title_text="Confidence", row=7, col=2)
fig.update_xaxes(title_text="Product Rules", row=8, col=1)
fig.update_yaxes(title_text="Lift", row=8, col=1)

fig.update_xaxes(title_text="Lift", row=8, col=2)
fig.update_yaxes(title_text="Confidence", row=8, col=2)


# =====================================================
# LEGEND POSITION
# =====================================================

fig.update_layout(
legend=dict(
x=1.10000,
y=1.025
)
)


# =====================================================
# FIX SUBPLOT TITLE OVERLAP
# =====================================================

for annotation in fig.layout.annotations:
    annotation.y += 0.015


# =====================================================
# FINAL LAYOUT
# =====================================================

fig.update_layout(

height=3500,
width=1500,

title=dict(
text="<b>Customer Analytics Dashboard: Segmentation, CLV & Market Basket Analysis</b>",
x=0.5,
y=0.99,
font=dict(size=32)
),

margin=dict(t=180),
template="plotly_white"

)

fig.show()

## **PREDICTIVE PURCHASE ANALYSIS FOR NEXT MONTH (how much a customer is likely to spend next month)**

In [ ]:
# ============================================================
# 1 – Imports
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.types import IntegerType

import plotly.express as px
import plotly.graph_objects as go


# ============================================================
# 2 – Prepare Feature Vector
# ============================================================

df = df.withColumn("Row ID", F.col("Row ID").cast(IntegerType()))
df = df.withColumn("Postal Code", F.col("Postal Code").cast(IntegerType()))

if "Order Year" not in df.columns:
    df = df.withColumn(
        "Order Year",
        F.date_format(F.col("Order Date"), "yyyy").cast(IntegerType())
    )


# ------------------------------------------------------------
# Create Customer Features (RFM)
# ------------------------------------------------------------

max_order_date = df.agg(F.max("Order Date")).collect()[0][0]

recency_df = df.groupBy("Customer ID").agg(
    F.max("Order Date").alias("Last_Purchase_Date")
)

recency_df = recency_df.withColumn(
    "Recency",
    F.datediff(F.lit(max_order_date), F.col("Last_Purchase_Date"))
)

frequency_df = df.groupBy("Customer ID").agg(
    F.count("Order ID").alias("Frequency")
)

monetary_df = df.groupBy("Customer ID").agg(
    F.sum("Sales").alias("Monetary")
)

customer_features = recency_df.join(frequency_df, "Customer ID") \
                              .join(monetary_df, "Customer ID")


# ------------------------------------------------------------
# Simulate Next Month Spend
# ------------------------------------------------------------

customer_features = customer_features.withColumn(
    "Next_Month_Spend",
    F.col("Monetary") * 0.1 + F.rand() * 50
)


# ------------------------------------------------------------
# Feature Vector
# ------------------------------------------------------------

feature_cols = ["Recency", "Frequency", "Monetary"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

data = assembler.transform(customer_features) \
                .select("Customer ID", "features", "Next_Month_Spend")


# ============================================================
# 3 – Train Models
# ============================================================

train_df, test_df = data.randomSplit([0.8, 0.2], seed=42)


# -------- Linear Regression --------
lr = LinearRegression(
    featuresCol="features",
    labelCol="Next_Month_Spend"
)

lr_model = lr.fit(train_df)
lr_preds = lr_model.transform(test_df).select(
    "Customer ID", "prediction"
)


# -------- Random Forest --------
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="Next_Month_Spend",
    numTrees=25
)

rf_model = rf.fit(train_df)
rf_preds = rf_model.transform(test_df).select(
    "Customer ID", "prediction"
)


# -------- Gradient Boosted Trees --------
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="Next_Month_Spend",
    maxIter=25
)

gbt_model = gbt.fit(train_df)
gbt_preds = gbt_model.transform(test_df).select(
    "Customer ID", "prediction"
)


# ============================================================
# 4 – Combine Predictions
# ============================================================

lr_df = lr_preds.withColumnRenamed("prediction", "LR_Prediction")
rf_df = rf_preds.withColumnRenamed("prediction", "RF_Prediction")
gbt_df = gbt_preds.withColumnRenamed("prediction", "GBT_Prediction")

combined_preds = lr_df.join(rf_df, "Customer ID") \
                      .join(gbt_df, "Customer ID")

combined_preds = combined_preds.join(
    test_df.select("Customer ID", "Next_Month_Spend"),
    "Customer ID"
)


# ============================================================
# 5 – Prediction Errors
# ============================================================

combined_preds = combined_preds.withColumn(
    "LR_Error",
    F.col("LR_Prediction") - F.col("Next_Month_Spend")
)

combined_preds = combined_preds.withColumn(
    "RF_Error",
    F.col("RF_Prediction") - F.col("Next_Month_Spend")
)

combined_preds = combined_preds.withColumn(
    "GBT_Error",
    F.col("GBT_Prediction") - F.col("Next_Month_Spend")
)


# Rank by predicted spend
windowSpec = Window.orderBy(F.desc("GBT_Prediction"))

combined_preds = combined_preds.withColumn(
    "GBT_Rank",
    F.row_number().over(windowSpec)
)


combined_preds.show(100, truncate=False)


# ============================================================
# 6 – Model Evaluation
# ============================================================

evaluator_rmse = RegressionEvaluator(
    labelCol="Next_Month_Spend",
    predictionCol="GBT_Prediction",
    metricName="rmse"
)

evaluator_mae = RegressionEvaluator(
    labelCol="Next_Month_Spend",
    predictionCol="GBT_Prediction",
    metricName="mae"
)

rmse = evaluator_rmse.evaluate(combined_preds)
mae = evaluator_mae.evaluate(combined_preds)

print("GBT RMSE:", rmse)
print("GBT MAE:", mae)


# ============================================================
# 7 – Convert to Pandas
# ============================================================

pred_pd = combined_preds.toPandas()
# ============================================================
# 8 – Visualization
# ============================================================


# ------------------------------------------------------------
# 1. Predicted vs Actual Spend
# ------------------------------------------------------------

fig1 = px.scatter(
    pred_pd,
    x="Next_Month_Spend",
    y="GBT_Prediction",
    color="GBT_Prediction",
    size="Next_Month_Spend",
    hover_data=["Customer ID"],
    title="Predicted vs Actual Customer Spend (GBT Model)",
)

# Perfect prediction line
fig1.add_trace(go.Scatter(
    x=pred_pd["Next_Month_Spend"],
    y=pred_pd["Next_Month_Spend"],
    mode='lines',
    name="Perfect Prediction",
    line=dict(color='red', dash='dash')
))

# Fix overlap of legend & colorbar
fig1.update_layout(

    xaxis_title="Actual Spend ($)",
    yaxis_title="Predicted Spend ($)",

    legend=dict(
        x=1.22,
        y=1
    ),

    margin=dict(r=120)

)

# Move colorbar slightly down
fig1.update_traces(
    marker=dict(
        colorbar=dict(
            title="Predicted Spend",
            y=0.45,
            len=0.2
        )
    )
)

fig1.show()


# ------------------------------------------------------------
# 2. Error Distribution
# ------------------------------------------------------------

fig2 = go.Figure()

fig2.add_trace(go.Histogram(
    x=pred_pd["GBT_Error"],
    name="GBT Error"
))

fig2.add_trace(go.Histogram(
    x=pred_pd["RF_Error"],
    name="Random Forest Error"
))

fig2.add_trace(go.Histogram(
    x=pred_pd["LR_Error"],
    name="Linear Regression Error"
))

fig2.update_layout(
    barmode="overlay",
    title="Prediction Error Distribution",
    xaxis_title="Prediction Error",
    yaxis_title="Customer Count"
)

fig2.update_traces(opacity=0.7)

fig2.show()


# ------------------------------------------------------------
# 3. Top Predicted Customers
# ------------------------------------------------------------

top_customers = pred_pd.sort_values(
    "GBT_Prediction",
    ascending=False
).head(20)

fig3 = px.bar(
    top_customers,
    x="Customer ID",
    y="GBT_Prediction",
    color="GBT_Prediction",
    title="Top 20 Customers by Predicted Spend",
)

fig3.update_layout(
    xaxis_title="Customer ID",
    yaxis_title="Predicted Spend ($)"
)

fig3.show()


# ------------------------------------------------------------
# 4. Model Comparison
# ------------------------------------------------------------

fig4 = go.Figure()

fig4.add_trace(go.Scatter(
    x=pred_pd.index,
    y=pred_pd["LR_Prediction"],
    mode="lines",
    name="Linear Regression"
))

fig4.add_trace(go.Scatter(
    x=pred_pd.index,
    y=pred_pd["RF_Prediction"],
    mode="lines",
    name="Random Forest"
))

fig4.add_trace(go.Scatter(
    x=pred_pd.index,
    y=pred_pd["GBT_Prediction"],
    mode="lines",
    name="Gradient Boosted Trees"
))

fig4.add_trace(go.Scatter(
    x=pred_pd.index,
    y=pred_pd["Next_Month_Spend"],
    mode="lines",
    name="Actual Spend",
    line=dict(color="black", dash="dash")
))

fig4.update_layout(
    title="Model Prediction Comparison",
    xaxis_title="Customers",
    yaxis_title="Spend ($)"
)

fig4.show()


**CUSTOMER COHORT ANALYSIS (MONTHLY)**

In [ ]:
# ============================================================
# 1️⃣ Imports
# ============================================================
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from pyspark.sql.functions import col, min as spark_min, trunc, count, months_between, floor
import pandas as pd
import plotly.express as px

# ============================================================
# 2️⃣ Prepare Data
# ============================================================

df = df.withColumn("Order_Month", trunc(col("Order Date"), "MM"))

customer_first_order = df.groupBy("Customer ID").agg(
    spark_min("Order_Month").alias("Cohort_Month")
)

df_cohort = df.join(customer_first_order, on="Customer ID", how="left")

df_cohort = df_cohort.withColumn(
    "Cohort_Index",
    (floor(months_between(col("Order_Month"), col("Cohort_Month"))) + 1).cast("int")
)

# ============================================================
# 3️⃣ Aggregate Cohorts
# ============================================================

cohort_data = df_cohort.groupBy("Cohort_Month", "Cohort_Index") \
                       .agg(count("Customer ID").alias("Num_Customers")) \
                       .orderBy("Cohort_Month", "Cohort_Index")

cohort_pd = cohort_data.toPandas()

cohort_counts = cohort_pd.pivot_table(
    index="Cohort_Month",
    columns="Cohort_Index",
    values="Num_Customers"
).fillna(0)

# ============================================================
# 4️⃣ Calculate Retention Rates
# ============================================================

initial_cohort_size = cohort_counts.iloc[:,0]

cohort_retention = cohort_counts.divide(initial_cohort_size, axis=0) * 100

cohort_retention = cohort_retention.round(1)

# ============================================================
# 5️⃣ Visualization
# ============================================================

fig = px.imshow(

    cohort_retention,

    text_auto=".1f",
    height=800,


    labels=dict(
        x="Months Since First Purchase",
        y="Customer Cohort (First Purchase Month)",
        color="Retention %"
    ),

    title="<b>Customer Cohort Retention Analysis</b>",

    color_continuous_scale="RdBu_r"

)

fig.update_layout(
    xaxis_side="top",
    template="plotly_white",
    title_x=0.5   # ⭐ Center title
)

fig.show()

**TESTING - HOW MUCH CUSTOMERS WILL SPEND NEXT MONTH (ACTUAL VS PREDICT)**

In [ ]:
# ============================================================
# 1️⃣ Imports
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.types import IntegerType

import plotly.express as px
import plotly.graph_objects as go


# ============================================================
# 2 – Prepare Feature Vector
# ============================================================

df = df.withColumn("Row ID", F.col("Row ID").cast(IntegerType()))
df = df.withColumn("Postal Code", F.col("Postal Code").cast(IntegerType()))

if "Order Year" not in df.columns:
    df = df.withColumn(
        "Order Year",
        F.date_format(F.col("Order Date"), "yyyy").cast(IntegerType())
    )


# ------------------------------------------------------------
# Create Customer Features (RFM)
# ------------------------------------------------------------

max_order_date = df.agg(F.max("Order Date")).collect()[0][0]

recency_df = df.groupBy("Customer ID").agg(
    F.max("Order Date").alias("Last_Purchase_Date")
)

recency_df = recency_df.withColumn(
    "Recency",
    F.datediff(F.lit(max_order_date), F.col("Last_Purchase_Date"))
)

frequency_df = df.groupBy("Customer ID").agg(
    F.count("Order ID").alias("Frequency")
)

monetary_df = df.groupBy("Customer ID").agg(
    F.sum("Sales").alias("Monetary")
)

customer_features = recency_df.join(frequency_df, "Customer ID") \
                              .join(monetary_df, "Customer ID")


# ------------------------------------------------------------
# Simulate Next Month Spend
# ------------------------------------------------------------

customer_features = customer_features.withColumn(
    "Next_Month_Spend",
    F.col("Monetary") * 0.1 + F.rand() * 50
)


# ------------------------------------------------------------
# Feature Vector
# ------------------------------------------------------------

feature_cols = ["Recency", "Frequency", "Monetary"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

data = assembler.transform(customer_features) \
                .select("Customer ID", "features", "Next_Month_Spend")


# ============================================================
# 3 – Train Models
# ============================================================

train_df, test_df = data.randomSplit([0.8, 0.2], seed=42)


# -------- Linear Regression --------
lr = LinearRegression(
    featuresCol="features",
    labelCol="Next_Month_Spend"
)

lr_model = lr.fit(train_df)
lr_preds = lr_model.transform(test_df).select(
    "Customer ID", "prediction"
)


# -------- Random Forest --------
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="Next_Month_Spend",
    numTrees=25
)

rf_model = rf.fit(train_df)
rf_preds = rf_model.transform(test_df).select(
    "Customer ID", "prediction"
)


# -------- Gradient Boosted Trees --------
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="Next_Month_Spend",
    maxIter=25
)

gbt_model = gbt.fit(train_df)
gbt_preds = gbt_model.transform(test_df).select(
    "Customer ID", "prediction"
)


# ============================================================
# 4 – Combine Predictions
# ============================================================

lr_df = lr_preds.withColumnRenamed("prediction", "LR_Prediction")
rf_df = rf_preds.withColumnRenamed("prediction", "RF_Prediction")
gbt_df = gbt_preds.withColumnRenamed("prediction", "GBT_Prediction")

combined_preds = lr_df.join(rf_df, "Customer ID") \
                      .join(gbt_df, "Customer ID")

combined_preds = combined_preds.join(
    test_df.select("Customer ID", "Next_Month_Spend"),
    "Customer ID"
)


# ============================================================
# 5 – Prediction Errors
# ============================================================

combined_preds = combined_preds.withColumn(
    "LR_Error",
    F.col("LR_Prediction") - F.col("Next_Month_Spend")
)

combined_preds = combined_preds.withColumn(
    "RF_Error",
    F.col("RF_Prediction") - F.col("Next_Month_Spend")
)

combined_preds = combined_preds.withColumn(
    "GBT_Error",
    F.col("GBT_Prediction") - F.col("Next_Month_Spend")
)


# Rank by predicted spend
windowSpec = Window.orderBy(F.desc("GBT_Prediction"))

combined_preds = combined_preds.withColumn(
    "GBT_Rank",
    F.row_number().over(windowSpec)
)


combined_preds.show(100, truncate=False)


# ============================================================
# 6 – Model Evaluation
# ============================================================

evaluator_rmse = RegressionEvaluator(
    labelCol="Next_Month_Spend",
    predictionCol="GBT_Prediction",
    metricName="rmse"
)

evaluator_mae = RegressionEvaluator(
    labelCol="Next_Month_Spend",
    predictionCol="GBT_Prediction",
    metricName="mae"
)

rmse = evaluator_rmse.evaluate(combined_preds)
mae = evaluator_mae.evaluate(combined_preds)

print("GBT RMSE:", rmse)
print("GBT MAE:", mae)


# ============================================================
# 7 – Convert to Pandas
# ============================================================

pred_pd = combined_preds.toPandas()
# ============================================================
# 8 – Visualization
# ============================================================


# ------------------------------------------------------------
# 1. Predicted vs Actual Spend
# ------------------------------------------------------------

fig1 = px.scatter(
    pred_pd,
    x="Next_Month_Spend",
    y="GBT_Prediction",
    color="GBT_Prediction",
    size="Next_Month_Spend",
    hover_data=["Customer ID"],
    title="Predicted vs Actual Customer Spend (GBT Model)",
)

# Perfect prediction line
fig1.add_trace(go.Scatter(
    x=pred_pd["Next_Month_Spend"],
    y=pred_pd["Next_Month_Spend"],
    mode='lines',
    name="Perfect Prediction",
    line=dict(color='red', dash='dash')
))

# Fix overlap of legend & colorbar
fig1.update_layout(

    xaxis_title="Actual Spend ($)",
    yaxis_title="Predicted Spend ($)",

    legend=dict(
        x=1.22,
        y=1
    ),

    margin=dict(r=120)

)

# Move colorbar slightly down
fig1.update_traces(
    marker=dict(
        colorbar=dict(
            title="Predicted Spend",
            y=0.45,
            len=0.2
        )
    )
)

fig1.show()


# ------------------------------------------------------------
# 2. Error Distribution
# ------------------------------------------------------------

fig2 = go.Figure()

fig2.add_trace(go.Histogram(
    x=pred_pd["GBT_Error"],
    name="GBT Error"
))

fig2.add_trace(go.Histogram(
    x=pred_pd["RF_Error"],
    name="Random Forest Error"
))

fig2.add_trace(go.Histogram(
    x=pred_pd["LR_Error"],
    name="Linear Regression Error"
))

fig2.update_layout(
    barmode="overlay",
    title="Prediction Error Distribution",
    xaxis_title="Prediction Error",
    yaxis_title="Customer Count"
)

fig2.update_traces(opacity=0.7)

fig2.show()


# ------------------------------------------------------------
# 3. Top Predicted Customers
# ------------------------------------------------------------

top_customers = pred_pd.sort_values(
    "GBT_Prediction",
    ascending=False
).head(20)

fig3 = px.bar(
    top_customers,
    x="Customer ID",
    y="GBT_Prediction",
    color="GBT_Prediction",
    title="Top 20 Customers by Predicted Spend",
)

fig3.update_layout(
    xaxis_title="Customer ID",
    yaxis_title="Predicted Spend ($)"
)

fig3.show()


# ------------------------------------------------------------
# 4. Model Comparison
# ------------------------------------------------------------

fig4 = go.Figure()

fig4.add_trace(go.Scatter(
    x=pred_pd.index,
    y=pred_pd["LR_Prediction"],
    mode="lines",
    name="Linear Regression"
))

fig4.add_trace(go.Scatter(
    x=pred_pd.index,
    y=pred_pd["RF_Prediction"],
    mode="lines",
    name="Random Forest"
))

fig4.add_trace(go.Scatter(
    x=pred_pd.index,
    y=pred_pd["GBT_Prediction"],
    mode="lines",
    name="Gradient Boosted Trees"
))

fig4.add_trace(go.Scatter(
    x=pred_pd.index,
    y=pred_pd["Next_Month_Spend"],
    mode="lines",
    name="Actual Spend",
    line=dict(color="black", dash="dash")
))

fig4.update_layout(
    title="Model Prediction Comparison",
    xaxis_title="Customers",
    yaxis_title="Spend ($)"
)

fig4.show()

**DASHBOARD**

In [ ]:
# ============================================================
# Dashboard Visualization
# ============================================================

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Top customers
top_customers = pred_pd.sort_values(
    "GBT_Prediction",
    ascending=False
).head(20)

# ============================================================
# Create Dashboard Layout
# ============================================================

fig = make_subplots(
    rows=3,
    cols=2,

    horizontal_spacing=0.15,
    vertical_spacing=0.12,

    specs=[
        [{}, {}],
        [{}, {}],
        [{"colspan": 2}, None]
    ],

    subplot_titles=(
        "<b>GBT: Predicted vs Actual Spend</b>",
        "<b>Prediction Error Distribution</b>",
        "<b>Top 20 Customers by Predicted Spend</b>",
        "<b>Random Forest: Actual vs Predicted Spend</b>",
        "<b>Model Prediction Comparison</b>",
    )
)

# ============================================================
# 1️⃣ GBT Predicted vs Actual
# ============================================================

fig.add_trace(
    go.Scatter(
        x=pred_pd["Next_Month_Spend"],
        y=pred_pd["GBT_Prediction"],
        mode="markers",
        marker=dict(
            size=8,
            color=pred_pd["GBT_Prediction"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="Prediction", len=0.35)
        ),
        text=pred_pd["Customer ID"],
        name="GBT Prediction"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=pred_pd["Next_Month_Spend"],
        y=pred_pd["Next_Month_Spend"],
        mode="lines",
        line=dict(color="black", dash="dash", width=2),
        name="Perfect Prediction"
    ),
    row=1, col=1
)

# Axis labels
fig.update_xaxes(title_text="Actual Spend ($)", row=1, col=1)
fig.update_yaxes(title_text="Predicted Spend ($)", row=1, col=1)

# ============================================================
# 2️⃣ Error Distribution
# ============================================================

fig.add_trace(
    go.Histogram(
        x=pred_pd["GBT_Error"],
        opacity=0.6,
        name="GBT Error"
    ),
    row=1, col=2
)

fig.add_trace(
    go.Histogram(
        x=pred_pd["RF_Error"],
        opacity=0.6,
        name="RF Error"
    ),
    row=1, col=2
)

fig.add_trace(
    go.Histogram(
        x=pred_pd["LR_Error"],
        opacity=0.6,
        name="LR Error"
    ),
    row=1, col=2
)

fig.update_xaxes(title_text="Prediction Error", row=1, col=2)
fig.update_yaxes(title_text="Number of Customers", row=1, col=2)

# ============================================================
# 3️⃣ Top Customers
# ============================================================

fig.add_trace(
    go.Bar(
        x=top_customers["Customer ID"],
        y=top_customers["GBT_Prediction"],
        marker=dict(color=top_customers["GBT_Prediction"]),
        name="Top Customers"
    ),
    row=2, col=1
)

fig.update_xaxes(title_text="Customer ID", row=2, col=1)
fig.update_yaxes(title_text="Predicted Spend ($)", row=2, col=1)

# ============================================================
# 4️⃣ Random Forest Actual vs Predicted
# ============================================================

fig.add_trace(
    go.Scatter(
        x=pred_pd["Next_Month_Spend"],
        y=pred_pd["RF_Prediction"],
        mode="markers",
        marker=dict(
            size=8,
            color=pred_pd["RF_Prediction"],
            colorscale="Plasma"
        ),
        text=pred_pd["Customer ID"],
        name="RF Prediction"
    ),
    row=2, col=2
)

fig.add_trace(
    go.Scatter(
        x=pred_pd["Next_Month_Spend"],
        y=pred_pd["Next_Month_Spend"],
        mode="lines",
        line=dict(color="black", dash="dash", width=2),
        name="Perfect Prediction"
    ),
    row=2, col=2
)

fig.update_xaxes(title_text="Actual Spend ($)", row=2, col=2)
fig.update_yaxes(title_text="Predicted Spend ($)", row=2, col=2)

# ============================================================
# 5️⃣ Model Prediction Comparison
# ============================================================

fig.add_trace(
    go.Scatter(
        x=pred_pd.index,
        y=pred_pd["LR_Prediction"],
        mode="lines",
        name="Linear Regression",
        line=dict(color="#1f77b4", width=2)
    ),
    row=3, col=1
)

fig.add_trace(
    go.Scatter(
        x=pred_pd.index,
        y=pred_pd["RF_Prediction"],
        mode="lines",
        name="Random Forest",
        line=dict(color="#2ca02c", width=2)
    ),
    row=3, col=1
)

fig.add_trace(
    go.Scatter(
        x=pred_pd.index,
        y=pred_pd["GBT_Prediction"],
        mode="lines",
        name="Gradient Boosted Trees",
        line=dict(color="#ff7f0e", width=2)
    ),
    row=3, col=1
)

fig.add_trace(
    go.Scatter(
        x=pred_pd.index,
        y=pred_pd["Next_Month_Spend"],
        mode="lines",
        name="Actual Spend",
        line=dict(color="black", dash="dash", width=3)
    ),
    row=3, col=1
)

fig.update_xaxes(title_text="Customers", row=3, col=1)
fig.update_yaxes(title_text="Spend ($)", row=3, col=1)

# ============================================================
# Dashboard Layout
# ============================================================

fig.update_layout(
    height=1450,
    width=1450,

    title={
        "text": "<b>Customer Spending Prediction Dashboard</b>",
        "x": 0.5,
        "xanchor": "center",
        "font": {"size": 34}
    },

    margin=dict(t=140),
    barmode="overlay",
    template="plotly_white",

    legend=dict(
        x=1.25,
        y=1,
        xanchor="right",
        yanchor="top"
    )
)

fig.show()